# CIC-DDoS2019 LightGBM CPU baseline

Production: full natural-distribution train split and exactly 100 boosting iterations. Resume/checkpoint state is synchronized with S3.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os
import subprocess
import sys
import time
import zlib

PROJECT_NAME = "Luan-Van-LightGBM-Parquet-Github-v2"
SESSION_MAXIMUM_HOURS = 12.0
SESSION_STOP_BEFORE_MINUTES = 30.0
os.environ["PIPELINE_SESSION_DEADLINE_EPOCH"] = str(
    time.time() + SESSION_MAXIMUM_HOURS * 3600.0 - SESSION_STOP_BEFORE_MINUTES * 60.0
)
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
SOURCE_DIR = PROJECT_DIR / "source"
PREPARED_DIR = PROJECT_DIR / "prepared"
RUNS_DIR = PROJECT_DIR / "runs"
for directory in (SOURCE_DIR, PREPARED_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

encoded_files = json.loads("{\"checkpoint.py\": \"eNrtPWtv3EaS3/UruFwIIbMj2o6TxWJyc7jYsZ1g8zBsZ+8WOoGghj0SVxxyjg/bilb//aqqH+wXZ8iRk9sPZyDxkOyurq53VT8chuHzJmuvz9psw4Ifiqvr7tWzH4Nndd12rAnW12x9s6uLqmuDrMqD96wpNgXLg6yrt8U6ePs0aG+rdRKG4cnJpqm3QZpu+q5vWJoGxXZXNx30q+ou64q6ak9OxLt1+17+vIbRy+JSPv6jrSv5u6yvrorqSj7WrfzVXvddUcqnrtgy+bvvi5wjkmcdwy8SDfm8oPa/1hXj7XZZh+PLZq/hkX/obncwuHz/TXW7CH7MdvhuEbxl/9Ozas1OTk5++PnVqxdvgpXENrli3Q/wkzVRmlbZFigRQ7OcbYK+W6dV/SGKg7N/D9quWZ4E8KdhQK9K4ZdgC4liAl3ipGjrTd1ssy6SkNrr7Iuv/pxuipJFOIEl4b0AfvXVTXp527F2GQDXAK0nj7/4Mvic/jLHzYsr1mILwYGEA4Ux8OuHorsm2iT1jlVR2FyGcZC10LjKS8YhULtrwCG4LOv1TbBcic9Jw7I80pCJhw7D0Em/w0lH1Dk2aMG/X7OP/Jea9zqr6qpYZ2WKSMPUb8s6y5fIHXNywJw6BzldkUAleb/dtbL5Ar62KKJZuy6K1cusbEEqWuByesNu29W7psdntssaEPOmXUXhIlwE4TKM44QDjsK+25z9JTSwtugoUIg90+DakyJqKaJmTGQR5NC2qEhlOGNpbj+BPAjODd8TQJJVXbK9yYsm4g9yBuxj0XZpfUOPHNOOoTRnzS0QRoeC3E7bfrMpPkb6e/4q+FMQJt12F2qioSAJ+fgQLjjRQQVWkjo+gVH8GNjBm/jZUlQ5TGn1hc2gWAEUIvehKUCWwv+uQufTpuxBWIbXdZts0GxF8juIcFVHMW8BXxu2K7M1i9QkDZ5YXLwGKtfNLWekeFgqC3EubMY5iOYC2XtxsSAapIbetu+1Z4vdjrSUMIgcKdagcfwlrAeIhgKhy4V66RUK8ApljuYODA/ih/O9AFDnF/xz3YCarOsmB44GkkqKI/gZWIvfeCvTXhQb+gp+BFtoQxmtTDQSoDur8gg6zhTbRVCxD2VRsVU4YvNQ1BpOp+TbYt39J72IpBwPSKyGn7HVnQvsNRhK6On/2NQfWsXn30KoJUulRL/PygJNspTpieK87hsUrhRRFlYLfI8lxepji1IB3yPO6fNQfQkvYr+gcBliH3ds3ZFRJw1osuqKRU/QRnSRg0MMAvpEzB7kRxv+DysFauBpkxUtC/6WlT170TR1ExmitQkFJgkqW7DtwXGu66rLAEf8u6j6um+DviqATPpYT5LkzkHt/usgtMDXly1r3sPcUCBWdwOE8+VXF/cBDFQab8++Wl7cD0CQgesya1uIyN4CnoLmEJZBhJbl2Q7ltW8xnhn0vN9x88uDujVBBmbWO5CMIdjj73mEx53PBoK8oiq6NI1aVm6wS7UprpaBIxuoW9llyfK0BmBNkbNlcFnXZfBPEgxgJP5lCQrZMIIIzgCZjT0i/uY8FABBVFRjRCIR71Eph77AdxuBoGj52AzcCwdtN7EgNyD6hQcTTr5UfnYw2mYf4WPXFAwlHp6UqAoAWgPo7IzagU+5zFqWtqAOVY5ANjDi0N9t4uBw2a9vGMZ4oP+sel80wEmITyPgkALD26TwGbpDmBPGCXwudpEFa9cwNPt7YfE2Plji7/CR5p4bdgWitR8ib7MXO+Ayegbe0nQJU0bYZGV5ma1v0glDCdYIoOIHWCyUKIdabXFVgYmBaHbdDTphyr7qpNqTQXYxBkVOX7958fb7Vz+9+DZ9/vNPL79/lb7+5t134ShRTJAmYcgZYqwRma1i7hcnRXL++cqQG5VD+KLY6DLoZ6rEE/lhgpFyqYu01VsJpK93+1R8BwA+2RM00rUEbZ7+/IeVi6s7e8dxhK8VKWR3MqGBAJsXmw1r2oCSTODqs1+e//XFuxHUxBwVauLZRI2/fAhqAqyNGgjcy+//y0LNNCwOhU58osH55KBsqky6Lgvwk5QEjWiJJIu09kiXCJVfRwr0Ub3i41jpJ0RjbXaFwENRwLgGNSt+5bQQBr0deEPDDOTwMkq6gTEuvOkrTOo5HwQCJl15ISH5kDUVaF4UnrZfB5AZZ6VRhWnYFoMO6QYXgReW5RIpl+Lu+z92DWh4090qZ86pTgpEnhiov3QILngjvac5y04P5VVPXje5rLv6qfGRfVyzXRd8T9+JHmhc4O0k0oUEEPFQfvnDNasCDyOhjSRTzEUaBnHJJKe24rgmgh5gQYC6widgBL/SjL/uwyj31yENcRLQ+h8QZ2LWKkKlhpWA3HuIg8AmmSULDo1/JufCfyal13PSqJvwTpPy+0d3stN9aFsQCnfk5wHDpq907OCpyAm3CagOE9fnCThxMBo6cg6fPfosvodpDIEkxTBidJRLkT4QAmvhmIfyjiGaEO5C1IJCsQxekEwh0/12A7OKrMPQlzJIlTk4gRrmDBOEW0xeYhjFPgkfkBqT72EKwcoRTqF+Eu1/c5B14XGylhnm8GNR5OfBF8HnnweRhHsGM/YCsu0RaNhpq9A5zR+d5sEG8hLQwOi0jb8OaDAqmVbBafJk04YaTxeyp0v0BZZaWQTzjxNZMl3wefgxozJpWzK2i6xWkAExLBwPdAUjgM7AkAZuVoZGhjxCAtd27Yg5dIRBWDnZ7WSmkfMZOAnLsHEowEOIBazoQIZbdNY+2ybEU2E1zG+I9vL6Q0UxGlc+UN1B0Y1Ejgo5/7TMviC0Ffr5iP0eww8pj6opj2klDigpd/cxvVT1Gpm4Q9badlm1ZhGBIoRiO8yn+d6FACBcBqJdiLm0fLy3iUNvvZTZZU2X8uxOEKet+2bNZLFuU1RZmRoE05LaATNImX8hKBA/gKBs+w6dEbIR7OUa3DbqCg4WdOC2+qvrIBOB2FlZbAusdLz++e07Sr3n050KKdkHEWlhJg7Ci6rpY4Q2ZckLGH/bRhahqZ0Rdw9DxEkzHmcrmmHoCi5CAwR+gtM3QdW/D12zJios7LBdprRGQNu/bmGaTHDVNVgDxCpytd7+IypsLWM30eN4tJkRGijDEic4VDTai69ewOzOw74pw4vF3pZ51mUrTDAj0YcKjlgI2N8Pq4Pt6i7Ev0FFIo0Dsh4f3+8HgSar7rvVk788fjzeUjh7lzbtrq5aZRhEIBBqWigZAvKofscnY4ASMqUpSH0K1qLr2yge89pY8z7OXx8O9pVRPzngTr09tfwMTDyZBq6UwtHKhIC+8EwAPTL45SX8F/q5wFnr/+ZxvCcuB0foyLMK6yXZIJ5pIB+K9ZZ113U+2FkwVMClvspxXC1yU3ZUDzGVkICVgvihwT5g2OUHbqpctydEgkzbG/H0I0BAbdFcTfjdu3ev35K0PK9zBoZitQq+fPwlZpBo2068gkY9yVfroAgAlWKGoixa27sQAOKHn+q3/fr6r+yWP3QvkQbh/eCB6kuKoWlppvV5ZZM4ouqlZ3pep2hyScsKXN9h9ucOQ+qnJ3AwPLWGEnVcTpOaPbbAAaAZB/LsC088vb3Ms6VrcrG9tJDU92Ix2K8vHlvG0pmV4j03LSku7ApZOWKeh0yVz0w5FDWsJk4p5fID0mVQgaekidYiekZlkpVWMlkEIJcr5OdhTCYZS1U2MLQ9nkMsMqJKOUSxnS+FzonKHqowbfErCibW7YWPRJ5RPTvFbwPBqErT9lsUY217Bu8VP1gDrYjNDNbwlSLAHKWkOAwg92XnUXUjEuZuJBgZRtacNXij87IIrvVxmg2lug0YTS0F4pJwJtyjSjMp7L1TON6HD/HhqBV7CnYerz7abo9RENneyrFaNhvDXQ/KhFMLAyoBejnqfXl8bKziYzn2hPh4RozsyVYTGEra6mHGYLAp1pVr7Hr0ORJjTowz1RCLYZLx3qB1SqzJg2wUQzS9h1zbiIuTiI22l6Ze0c52dAP1xt3deKBnTmLqtDFpr6Q9oo58l0MrI6WqAwzPfmDVVXcNVD97EseYFqI5HRcqrpHf/+ysZNA4VP9FU60vWouoOfSjuX//m58MV6w7gpXQaz4nkVaCkUP/C6p+sGwr9vEYWjCfqwB4siyDXeW7/GgHjuyNWxLSNeeo2PCHbFjpuw7HeQqSQjD3m5LR3YIecO7OQVorE675IdL19rtvzkA8ZgjYWLo5K+Wc7rLGU8+p6aebgnqzT3DXxSZbT89AD2WhUzPRcXEeda9/DJ5l6xuYNa54bnfAtcuiLLpbviJQ4j7V27OibXteY91m3RkYEqzN0YIt+po2+U3rTuBy1JacT157GvOpasRj/OrUfM1jCc2xx6XBaDdVEOa4ZwV8snu2XfOhaRx2zQYT9rvnQxOYMmXhlq2OD3XNo4Zz2Gc2zzXj/rN0johpjKG++zytwxTUi5MDZVgNKjBKEGxFHznZqIF4zxOygZ9/fvzYL75TmGzQYgqP50Wcc6LNTxZp7kN6hhh/wujyk0aW06PKWRHljGjyN4kk96A/SS4/dfR4OHKcFDUeGTF+6mjxX2lh4oEBoAz0PAHdjKoI0bGcU91Q0p6zkgHDhcDzp3FXezAwX07aG/G87sucql18QK1QNfhBXn4lXSBKeSppg3PGpVJaKR1qWXja4uwOT7gl+L8vqa79UStxOcTxUuuwrRHFVlob3G9j9BKz1mvcpdJWTV4RXQRGBdqY+nhk+OJj12TfNFe4eKmWdJbBXcjtLPyUqns/sn55RHg1yULbUeHRdXiDEvEEZL2R3bl0hcITXigviMtbVhRIrtO3QiaoOtsmbsJ3evznMYO8UmvM9d5jGg/TnYIkufgxWVq1XuPS6uHQaFvknNLV8WbPYdy3pAAgwBx+uDQ1IcRFwqWpEHvW4iXbvgXDv8bddqvw+c+v/x4+QPpnhpF68Hi01O9ZV5gQBO6Xdi1Y/A0k/aV0+qNSrq1H/Is7//8bl+919+POS0iV8O3HW9PDvv6gnzeyXPTZRBBrTLWKKZfOxcqgscTvOfa7UPv/xLEpsZP7U+4GUL3M7QeI99JdBt+/x/0lzErubJDK8UbuYAT6iAAIh6Q1YZj30+WjR3caz+4f3XnUxIv88aegH3gSWnJRQ9LdFzp9lfe4vRaHl3YPMm1krdNaaJUoBL+8+YFbsxE7tjegn1s3DF0aTHHu+p4PM+19eMI7pw7jP2s8oSRsJMpqxN9jlcU4Rj+SLk84Gj/VuxmrsYZdHIkhXNupzDwGXJRZKKRiayPLTMR9SHt9FJB24LIwnR5ODG36qiwqPC8wnMDA2AhPzPm2fHmPDu61+sbpp0M2Xd+z97vsEBPbl8VeZM+u5SP3gykiztkU5hqIqVvCZu7kUs2JzfHJoc0JWKXbu5traDBxM5dARB3MbCM59nn4rM5vwwt+ZUsMQZVxv8lwsPy5OoT2Y1ZlV0zeIqMfBzfO8w0EBJLigk5T1x2JNsgxhTeqwRaGLClk5KGQuaeKBk3HD5kPoz6d2GrKqXTewXM2nWvWMCVoTwdptVf2SXA1PTwIrh7MRs5UMTDBjd3OBwt6+xRaimP/kSLBwjdPq2dG6WJKsOk+C2dqj4JQNGr6isQ31C1XW5f8U1rk6oAb6RfLJaftg/7GITIRn/AOJ+MLmCzXRHlbd0pl2qeJaUDxjX4czcE/jpWJg6bSatLxONsOiqHw9Cf/KTJFUvVQbO/F4xxZA7KG+3phlEr83GV9Swc0Q1pkJiMBTeumC++9U6XTFDTMechJqp99ljibPBvxOXw52tN8/uFunv4dOtMN2FHD34ZAFpFoJB+NrFORg6bdp3fG9Vp0InyDj9Fnp38/3Z7mZ6ffnf6IhxPNo5GYTjhHI0mMXxtH6vWjHqb+8H4D1BZMN74T/sJz7pJ/kWcgp1ytQivmSonvDNpJMi3FIKZbk6xailGtr87o4XLkshWrI191wf260ENdezY0Gk5juVdh8YksfPLrqsOgwp7t0fDR3TdsgVwEE62GJ4vHPatbRqxkI2JiHWjgIqVOZwoJ450044wA5T0Qsg+YYu0ceIjPHYQfqE0cA2Gexyk03CIA79zTBmRLxXFgISvGkI98w8Wxn+6eaP0o8AuNGEM5xMrlB5On2h5zjQV2PmjpHjBBOnBKRinpPnY4tz28tRrvnftMnLZ47HbdPtKvNLKwEU0IE6PZYTxSqoLeqiSgjQRcwcrDmZYy9YqZ0svtPVg0Nq6spRE8T2ToMabccxxWQItJwxVd4ko4F8IoYQ291Y4PDKjwijZO4jzkb0XJ+sJ3plgtTn8r6AH6b6Kr1qjzmvFMbJt16+vAq4dj6Okz1RGU7+eiaFyzNQtBJQikAftscgdOgZ2bqfWCX1pnvrxwM+7f2YJTJu/qweEs/mHm8LApJPjRFMGee6OQ6Lx39JmGZuxaO34dl5BYN9a5cBNoaruQOGpRXQbBwsDcsURYj/O0JrTb1P1wyS/h5ZeC2rZl6RXYoRleXrpt6ZpUC+yGZXRBbwv4bjNfiy5rsMJgxpwLQ7hlgDokyX6tohvLNO2Hvg2/wo+C8R1rNpCw9hiJ6YWRaVqmZca89YimDXadC6nbWkrvCOg5CwuOHInTY4KbifM9iqdIqhtyezaNkG0PVjZhHE9F93bacpaQEFObyCim8ncQIFT9dhh/tQcjs/wtxpX7ov80rvv777DkxTw5jwnTDE/21X95u4UGMnYcOLmhlSE3o45b9lm378e7wMfQzn18F8guDBQWOnDLy7gZHzf7YB1bnrE9sZIzHjzgYi18DEu88/vqcpuIO79DX2tqaEWkZrN6Bwpd/Ar9l1r5bMAJeJX35djnA0mqspC0VUI+TEhWnXdWH9vSifzWfm2nt5pxhR7ak9XOY2qhveetl0NqR5MecGnSanWyQq6lN05DaYr9owHz+GXdgggav60jr7NqCGhVphQCjJDJ1ey9kZJTRBB+eoDpVg6csqqoXaERVJe1pA2uUxMJcV2fLL19AZC818WD9UbQlkNxL4JdPn6a35uWikhB19vTFiWL54thPO1wxnHFEB3qUflrPGEQ04rNTEnnDACGcQZ8NMOTwOvVhwcUSazIdQddDdmKTNEZizs9MVNwJmMq/bC8HkOk2YaWj8eiU7f6yOw7Sxf7o86xlZKJYZzlbGdly6bTPdD10ztfN9G1Tv8bdtfuq0wj9lLG8dMmkceYxnl25P9V3KoPkwvhW/5ENO2qmZHb+RMuzzKD2Pa4L5XAGow3hPFVYPSrYfkuRh5ab7PboK7K2+CS0WzwuoWgu2b63dp8gOGuc61GpO2rGNF9jxhvQo1k0mPas7Cc5Sf55yympEBO8uOQ9+SIrSbH6ZsGalxgN4bE3ulE4LesubUNrY0my5CNpXLzp3e1yriEZQ0KcUV+QhbbnA0rxyzbiP2l45O9kwPbl8npeulxu3xCOe2cJrQ9//wG/rlhbCcucX8sqkbeGPI8pMXJFNsbsaNeS6I3qBg1uu1IjZ5clfVlZISMn9uRFu4Pq8ucH6MCMOfLMxzrgv8TGYAkLWTTJyu1LvNh89H/AkOdkec=\", \"config/data.json\": \"eNqNVE1PGzEQvfMrIquHBAEJ4aMtFw5FlSr1gERvkFoTe7Kx4rUX25uQAv+9Y6+dDwpS9zQ7fn4ez3ue54Nej0kI4DGwq94z/eYEl8pRhg0XUFUah8o0bRgKJaS0fjw6/XrcgHtsMRw36I6FBu/Rs6OOYKY08gZCQGciyeHh8PAk4wsmgKswcGF1W0eQabV+b4kLMFJRSUR/1btnP2GKmh31mC7Bt3h4DEQJOoIY/eqiSWZOe/jM2ZrHGg3UyNWM18p7ZSriD67FjPVQN3QNJT8oZLMez7lLP70fN/Hnu7arnZCicn7lbNu8z5cAPeItu/OemLqzrRPEfhspb9AHZSAoa3ImL99aF94CSu7W2WDpXDZJpJO9O3pOGqaGZB1o8TUimG+02jFGcKCiVqOTz6PMsAQd70BnpfzpRdGQitjPeERJmfFofFkyxBewWkeLQBss79oDK3BYbOLwsVUOOWjNs8s4gpjzUlpUbFNv47BxVmCRM9eNTwQWKnDpbGl/anrpQ8onN2TX7krC+r9f7vnDg58MZqRNv4uvlRxcf2JH/4L63okXnzQZFHBfNS8NaTH4aI/04UVuhfvvjUHVtItkfHfVq1ppDW4eAgH2pDdtjU4JbttAL5vLsG6i/IyuCOFsXPpvwHBlZryx1MGkFLVYg6BOWa+CWpI0RnKDFaQfwiqjwpqvVJjzuHuB2KRgZh3XqpqHaloXei/IP0kqZqxBtlGyK2srobOrzqY0RmJ6fDk+PT/PLMLWpDyJnlzI/vggt0w11tatt0xkkDSf8vvdEGf/X4zoK+MiV0vrsOB1q4MiI2EcjGcnBVTDE4clKJotkQ5otDgQmxfxZbSpBFq5+5qmIBZo4ptgEsl0NTWODCA4VWhnmw49ks+RTyGQ6WOxb2qc0cnc0jRx1vvuVXC7RKeh2Z9nBZhn0IfwWO7B68FfIJLkyA==\", \"config/data.smoke.json\": \"eNqNVE1PGzEQvfMrolUPCQISQqGUC4dWlSr11t4gtSb2ZDOK1za2l5CS/veOvet8SCD1NvY8Pz+/mfHryWBQKYgQMFZ3g1de9htCkeedaryCutY4JuPaOJYklbJhOrn8fO7AP7UYzx36c6khBAzVWUewII3CQYzoTSI5PR2fXvT4gonga4xCWt02CWRard9KCQlGEUti+rvBQ/UD5qirs0GlS/AlXZ4CWYKOIEW/umjWM+czYuFtI5JGAw0KWoiGQiBTM3/0LfbYAI3jZ5B6R8gun+75mReD71/T4pu264OQo3J/7W3r3uZ72B2bzY4UBMEOZ7kMm04mE07+TYgqOE0HZYseKDk5ufg06RmeQacbyHb7l9fFYQzxeCcgqsw/vSk7zBex3qQCQhut6MTDGjyWInp8asmjAK1F3wMCQS5FkZb83Ol1Hp23EovZvW58YbCkKJS3xZxsSfEh7+da9T2VsznH2eHv7YN4fAyz0YINHHbxPanR/Yde5hFoGLzcBtt6iaMCHpLbOuvj6L0zKsStYs/IZDP/+2Ckhk9xGd/MBmpIa/DLGBmQ8+XJpm3QkxS2jTx3QsWNS+Wv+IkQr6bFfwNGkFkIZ9nBXCm2WINkp2ygSM9cGqOEwRrygrFkKG7EmuJSpNMrRJeDhfVCU72M9bwp9EFy/+RSVcYarHaV7GTtS+jtumtTHvK0fXlzdfuxJ5G24cJzzXMTVn9CVHuiBhvrN3si7o/8efTDteM9aP8yyr1WTsNKNK2OxG2E6dO6uiigBl4EPAPx3Cc24LH3IHfzcLufJWjV4SzNQa7QpImowhO38q7ju5WYQ+QuT/IYcr1XteC7hOXZ9jaEbgqEfUavwR3/LgXY/wjvwpPAk78n/wDAF8Ie\", \"config/orchestration.json\": \"eNp1kT9PwzAQxfd+iigzEfkDEu3I0qVITKzW1T45pvEl2HcVCPHduSQF0YHBkq1793vvyZ+boihP4P2A5oSJcCh3RemEPHn5QGofmrrZbpvbQYCqs54h+J79MVYTpDdBrnzgXo7VuS1vZpgDhoxs8ijJ4j80G6xzY2719cuZMFV2gJwxr6Qpja9o2RDEhXOYI7zoOcwR9o9P1fNldX8dgSF5TRAYE3AYSZebul5G2fboRMvGQMLqtCu6dZLQIrGZJPfGCyT3R3J3v0qEKJA3PULiI4KWZFBWr1VnWdMtsgjvIUo0WiSruwFmjBMviqa+ljB4ArVNqNe0aFbI5U9+GBwijqKGaEdyS6aurevN1+YbINSWcQ==\", \"config/report.json\": \"eNpdj0sOwjAMRPc9RcWaRSifBZexTDBqRJNYiatWoN4dN+Wf5cybzPhe1fWKE52dFRcD2LYPV0hxyKtj3eyNvvWMeByd7706FjgBd1Ego+eOCmjMm6SRO3QBNPFFfH7KLfJvzeZlMSXfC5YhiZhQSrR4EhkuqvSJMkgsC9TcLskTBdt6TFeV7irMEoptIbsblYpmt170AbWEwQml0rSse5qeMGuDpyD/hFFiWi4gOpejm0M1VQ9AamKs\", \"config/train.json\": \"eNqFVdtu2zAMfe9XFHletsRth21v7ZYWxbq26GUbMAyCbDO2VllydXGaBf33kZLlXFpgD0EQHlIkD3mY1d7+/qg1+g8UjinewOjT/ujCczX+jp8LUdXu7OTb+JqbRw9ufCZc7fNxl43eUGCjS5BDmCTvKm8iBk8tGNGAcsxoGRxybkEKBazQyqExOlqAEtFskr0Pv0voRBH8i9ZHF+Ublmtt6SmvyHs6mcQs3Mgls063rVAVAnMuLQRINDmXXBXAaq5KGeGR0grio3PgzhtgWBM2L7Tahr0FxqVkznChMO/CIu6Mh43OW254Q/YV2tAaasREzC3b0EGVly48h6DOiWTRBaDx0olCcmsTLLEVRbGGO3KZvJ1Mpj1G/SPeAeU6SNaGP7ESWlejcTwYsdiSO87wG0PmgdmUgjd5yZmcxud3rFmwbr5TUedOM9tK4bZiKHUuiLHs6Ki3JT7nhic6p0NAzquKmvsPCI+UJiUBZ0SBhl+RLiZ1JTUx1vPHwBhtRr97/7g4A/dpe3r+XG2Al0Tg4eDvwGCfAmdWbAyXetGmoDWVbCEsvIbhQiRsWLmXy7zR3Etkl7BXXMIkDa6vbl5Ba6xcV7iCrNVYqhV/qZxp9mFglpb4EdXsECoZupbbvRAxZI1ONFIiaJoSRGuvAFCwSCu18UQHJtdWuCUFouk5ihgLt+DW0mgNoFqwiNBSKQyNSHvXemffkS3NCnfAC5IlnwNroNFmyfBAzYXcmYPUvAzEOlJMRflHbbxTyBV+ofRHw1jib5ZzV9RJzYeTj8MsDOBM+SKU1+cZmilqKB5aLdRGP/gDTMdlvEiBtUS6RaEy0TTe8VxCdGBUv91toMD4B4B2xycbMluwNqplNQhPNDi0WnsTkmbDrOkIshxwPZE3obyL12JbtLFaFIOwbP248lKucx6s04GiFnaWxrdEPesHtQOGJCTckP1oGKrDMdL9x6x4/QNh69JzXzzg1EB1NMTbA3Zy//nr7C5ND3dnLp424Oub2en5z/XGVKSeHj7+cctuZmfnV5cJR33KnBcP7KXjl9np8f3FXQoYOMBLU8U/jFW6zh1Iijq/PL0aTnb/h8fon06XSQB7z3v/AB78NWQ=\", \"config/train.smoke.json\": \"eNqFVVlvEzEQfu+vqPJMIEkpAt5aSKuK0qIegISQ5d2d7Jr42PpIGqL+d2bs9eZoJR6iaOeb85vD64PDw0FrzR8oPdNcweDj4eAycD38jr9LUTf+/PTr8Bu3DwH88Fz4JhTDxWTwigyVqUD2ZpK060IlDB5bsEKB9swaGRUK7kAKDaw02qMwKTqACtHJaPIuflewEGXUL9uQVHRQrDDGkaugSXs8GqUo3MoVc960rdA1AjMuHURIqIJLrktgDdeVTPBAGw3J6Qy4DxYY5oTFC6N34eCAcSmZt1xojLt0iHsbYKvylluuSL5GGUpjjhiI+VUbK6iLykd3CJqCSBaLCKggvSgldy7DEkvRZGu5J5XR69Fo3GFUP+ILoFhHWar4I6ug9Q0Kh70Qk6245wz/0WQWmc0huCoqzuQ4ud+TTqJ0209NlXvDXCuF37Gh0IUgxibHx50s8zmzPNM57g0KXtdU3H9AeKAwOQh4K0oU/Ep0MWlqaYixjj8G1ho7+N3pp8Hpuc/T0/HnGwu8IgLHvb4Hi3UK7Fm51VyqxdiSxlSypXDwEoYDkbF+5J4P81Zxz5F9wl5QiZ20OL5GvYA2mLmpcQRZazBVJ/5SOuPJ+55ZGuIH3GaPUMVQtdqthYghaVKilkaCcoAk7TYANCzzSG25WIAtjBN+FacQZU9pizFzB36zG60FXBfMItZUCUs9MsG3wbs3JBs6ZeaQW4ajEARtJ58BU6CMXTG8UzMh99ohDa8iv54Wp6Y0Bm06V0gZ/uEFGPTdSd+s4L5s8lK/HX3oW2IBW8uXMckuTl9S2UA5b43QW1XhB9gFl+kwRfIy9w73lQmlgueFhKTAKH+3X0CJ9nOAdk9n0kd24FxamnW/f0Jh7xoTbAw66VtOt5AVgFOKvAkdfDoau7ubssWdEI5tnI9Hm4hHm2CgqYBqb9JDS8yzrk/7aAxC+wvbG0eSFaNnAKPiIxAJ22RWhHKOXQO9oCbeHrHT+09fpne5ezhBM/G4BX+7mZ5d/NxMTE1L1MEnP27ZzfT84voq45ieLHg5Z88VP0/PTu4v77JBzwIenDq9G+t8pBcgyeri6uy6v9zdu8fowTPxdSIPB08H/wCHsTcQ\", \"data.py\": \"eNrdfe932zay6Hf/Fbz88C6ZyoqTdLNd7So9TqJ0c+vYWTu9fXu8ejy0RNncSKRKUrFdP9+//c0vgAAIynLafnk5J4lEAoPBYDAzGMyMwjD8WGXrtMqCZZZ+Ti+z/TpdZMGb92/2374tz54fPPtL8DGtftlkTVCvl3lTB4uyCo7yy6vmh9cfhnt7n64yfhPkdZDWdX5ZZPPgIoNmWbDI0mYD/8/K4ktW1XlZBEbv4G3apDVAnlXQDl4O907La4ACPYoMOgQX6TItZtl8ENTpar3ED9cZ9sZPAKkoq1W6zH/N5sMgOEqryyzIi/WmIRh766qcZXUN6JRFpqdRldfBZVVu1kHaBGnQ5KssSIt5cF3lTZMVMAeez369zmb5Ip8FQJ+mHu6FYbi3t6jKVZAkiw3OK0mCfLUuK4BTFGVDc6j39tSz6hJ61pn6fjlTn67S+mqZX6iv/67LQn1epc2V+lzW6tN6mTZA0JX6Xmmg9S+AavZCf22qzaxR33BujPGsXC6zGeGnUH5TboomqwbBPFukm2Uzz6EjNZ6nTUZkkZbq+4AA/grE5HZrwBWmoZp9RNTpRXO7zotL9fywuB0E72Go9GIJMD6ka3w7CM4yWA9YXU2wYrNa3yL9i7WeOKxMinwVrOf62W1a4SLiw9R5OFzLKuPLX/TLetPky729vbOPR+8/JceHHyZnwTiIwqZK8yIcBOEX4KI5rR9+a7K6CeO9D4dnP778FhoW6+EmL5qX30YHN++cP/HeD5Pjyenhp8nb5Ozww8ejSfLuPfzz5uTopw/H0DlMmHeTRQ7/5POw2+H05GdPe5gONZ8cvzl5C42PDl9Pjsx2y/QiWwJP7s2WsPEC3MnM8UDej+mmzk6RwjVslugU1hrWbgI0quLRXgB/gJ1P0xx3R7qAxYGtMN/QEgV1ualm2T7iG1wAl8zT6ja4voK90cBm/zG9vMRGOA7sZ9j1RZZWy9ughB07pE2yBywVLMt0nsDGX+SXEXLKCJkz+L/EJnGw/ypAhjuHZwPkkCkjxe0TbA9TxKbUN6aX1zk8NVoMy3VWRGEFSwZ8VM5h2uNw0yz2vwtjZIArYJ5lxoDxT76wetebxSK/Gc5ABC3K5TyKgzFQdYi7MWw7tVgBQvhuiBOLGHasm2XLOrM7NdWt/YBQYIa8TVdL6112M8vWTfCeXtMq4QTgaRdEhYsWmAsahf88/HAkWMIaIhuDiPhlk1cZsMUtvv1r8F9nJ8ewVNkcFqwE0MD7sPWBgnMg3i1QjLYuDOmfOqI8ROWQdOYPdAXpB5yQF3WD8jriXgNa4ridAqP+3+lyoxB/Y+NcApjVpm5Af4BIDcqLf4PQCnkUmdAccLkL56w5cLOSsMYPa5P/8UG5aUAX4KdVtiqrW/yUbubQ+p4grnJqCgBroDrsEjXEcJ4vFlmVtVOJ9Uyl05ZJLcIPAthekVoE8Ci4EyD3amrYoAY8zhdA3EbGPJepTc+LdJVNY9K8+BF0XGBIsqlCLS1uoy+IR/C3cXBAzfkrtOchYlaarGaGeT1blnUW1ZtVJO8HwbPhwSBIL+qkKZfjZ9n+s+dbF5Ak6NNWfD5F2akmpJZyXdZ5k39hPQujBU0ZPAs1SdV07QWcDi+zJgrrGcCGr3HwH7A7C1A+4TZ8Pl2BQNIGxgUwCfTOGBOcOILLUNNkFWh2sU/qFhcQ85r8wj/T8xBEcZ2ssypBUyCEpUACb0ODuw6tfh1yaMYGHAqhggjPtClX+SxBeZPMQS2CFLzFfTdS6rMVnKjA6yYviPyjVrweA6kYReM96sesaIarz/O8ivhLPf5UbUAxZzd53STlZ/rKqDUZCiQU/2MLCkrihOVnZD7nR8E3IEab1To0xLaGJEL7elehTSLXJMFAmmD/Go2wtJ7l+fhdChJ4AOsH4qwZPx/Qnk4+Z7e1MR/8w72HaO5lUfivQrAs6yGw3zKFLa9xtUgby9LMYdugqkvE1CC9XkcokRIgqqnmBmgigW4t6CGtyhJofI7vROGRyBNNp0BYUhUbDGlp6qizE9/B0Mdl8w6VtBI+yqgGQCBwQOwF8zKrCRaBAfGDMJXsIexbEUiaFyUHfchZJg8vl+VFJHOJEbM1yw+aexRbCBPAXRA9LrVNzkiAUJpdgYC/k5H+o7oPoD3YJhbGsl+oj6xJXoC4BmNjuVkVEf8HYlYZmLhXYJsoaZzN1RrhFoHnwBAovTK3Cy1Y25LnhKYCKyF4I2PFhhFBsNVzoiR/RloKZve24COMNL0WSCIAz+OQDGSAqqk5WGzaNtwTxF+L7BYd9UYDVAjetWMg5a9TZhoCG7YjKfLjY6FI1dKQptkS9MFpqaYPzwpx8czMxUYeYEthDlABsBESOvMJj9T9TOJjBqvNtN3H+PV35gqLmAnu4x6CprNmky7JZrBJihaCRU3LbrBgT01CC0BU0elyGXGHluwWFG4be5fiXCPh6TI11+h8qlaIDhzqgARmGBx3QT8m+thAJAfVzAO2p36YvtVYi/DwX/9Ca++pLTIAwhCt3OTiFmgZyTF8eLFMP2fPLyLDm0CqCcCIYkJL9hJUQVLD2/F38ZC/RvAivMjBOJGJJGSxrfIbOCqS6QVcBIdHPEFV6S3Nov3Kk/kV5sBNh2kNB+cs0qdNFFnrW0N9kSaF11lVgaEN6gv10DjMLwFzsCba5UCg0a/B/8F/Xr0yzq8vDmJgiCfWifb1uz999+2fX7599mby7eRPr/8SPwzm+Z+7YP7y7duDb//y+vWzFy+ePXs2ed2RGH6EniGk/xXwWVvxA1IxwRUADYPrhSslC2aTdCCmprFZyYSGnVxnKOhhFC/ZYY2IfcwVY/B9y0AGRgxTaJHHIURabYoc9TiBNSAQMi+/jYOnAZv2z588ga+ieWEVUSTSC57H+cGUX8JhoWSBSa2+sVs9m1p8DQPB8bzKIsLib9xnEIAh775hsGDlD4LncWwgChP6TnExe/HI/2CsRCTbk2g6CNg78fsshkyjb9llqEcsjKAaK4wYATU/1gQCfFGBgBoF6/kQDad3+G0QWLrCZxW4MwCxWhb5jAQyQRwuy9n5aEB6IrLAxVM1kRDA0dFmCPguizQK/3Z8+MoWWYAX+q6GiG3C3rCEj8WRHpOt3htlAs9AMlyWFUoqEh3DpkzItRbNcdixJpOWWhoSu5RYdI3wdKFMoJEyWQCfvAZUqcnWk6E6BDfolm0CgkwmP7kdM2VS8PMx6UcGOkSyrCPLqKRW20abrNbN7UNjCU3ptXabTW5ANR2x9/sQvQNlpR1kb/P68/5FOvsMsiLDdgH5D4ISmCgrMmAuePHs+Xf7F/CQ3XbB+7c1KVH2MYtMIbcYn8WA3kkOuzFJQH4sFwN0r6Z4UBV1xweHC7SFcQvW7baxrR+r2yPPdUJZG0T3kNEdZlPAcfpz1ELBKQxBphTs3MCFZI+0ehhZAHo7wujZbIMnso+nhz98OAz+DTZBARy5AnEw/vnwKNy9a31bzK6qsig39fj45PTDTp2tSYdvTieHnybBp8PXR5MgxyMlnNjhiBJ9hq0WfJr870/B8Qn8/enoaKDe3wavj05eG8/5WuT98afJD5NT43loj/Xx9P2Hw9N/Bj9O/kngW4CgGX9+/+nvJz99Ck5Pfn7/tu34tfOZfPgokyJGTojNgsiegYHQrgi07AocgD6U9oHFcW5j24/i39qE6ZDZKjH69rhTNE7iRp9VZV2zIgPcDuw2Ipm3NREw8w28RtFaJ2iG5YXRvt3Z6XzOKMrexvUkC3ZgcNFI34ackyE6HRiqr2+759gFZBPSFz5GLbh22tdX6LLHbW7TlDAa8ZmlgXmocad4fJg+7LVGOz5hL2JxmUXOIsbdDnrQYbpGD3MUFdlNE6k5xAPjjGe4v8+ack2kIVdWB+oaxLX1UJQDT6/T/KICoW497RUcbydHE9ge705PPpgbI4x36b5Cv2v4/vhscvopODkN3v8AQmeC+/7EBKY3WRz89+HRT5OzIPo+DkXS2wMRP8pG2ml/0x4/g0m8+RS8Ofnp+FP0JO7MJjg848EcAUSd/+vk/bEp6KDt56K8LoKTY/4wRFYefx8cHr+VB2o+Y15pLUM80H/++wQowt2I1f/26vtw0GmnxB9OW2+IOLYbgrWUwXiwOaJYm8t6ifC0+f8z4cZ/DN1gJ5FmwxswFnhhd0P1SdVvxsyxWzv0yU/ojItmi4POZdoWge0dfffF9+/bdkEdlSyiOvYtlvDR92br77tC5WvXb+sEZ+VqlTdgmGldhKqIaV+LMtrhALeDHmpZfRxEHG8A1ufscxS++sc/Qp5Fe/7CbzySOKL4C3uWCQFHaxsqVHGiqTtNFJ3JEmuoufpcBQ9PDk7JcNKTG3/6vJNTIN6NOkIcPuXE9v0cj7aFFjS7XUhRZfVm2QgZeLuk13AgGAUXZbnsvXkX6YlbFfUsOUd6zSjTSOIhdJ8INbIxLF059tlaJhwEUDU2DsCz4dusAR7Ha4EedO7NkxeQcgXTz/E7ByyF7ggmwp4BOmjec4/tsOVoeWeb3ausuSrn4SgI6fSYiBG7rvJVWt3i7ZQjDYTlYV+YGCTo51um6EKAAy2A66FELywtfC3Z6wXXI6cd0B0i+VHsNOvFkNcGJJksDEKwOME3vqeTtbh2Hz/g9qzu76XHqKHnXVfo0wU4HOjBbJUL/qS9DU/yOsnQM7F9On2A8C79MSDMgb8Szn379b6VKRInAEvqEZodVUSNY+1gwf1VrfICTh/5rMfRQt+DdVWWC5KLGJpi9NIKdb8p91l2LDYFD8dYUhgiGn8iC7RRBcBaTwzFKC6v01u8bISTyTy4uKWgJu6aZfOMb5GosRoC75Lh0IVOPjpuZG2cY1MGzXUZqHgRFSQ5DDAOgSClX0pQcjAhEh37F/lyCSD3MYDt7B9HOd2YzbMbObutQXpn1RfymwFaJDX29JFApn65SeEg1mTZUJHvD9f61q3a76x0vbB/gxbzi2K9+8kp1iemLa5LmCHIU5woZkiISXvEtikQlSA8eFB09rXcLpF7QROdQiZUZGpip/XsKpt9/q1SrUvNXYVZT89dZVi3+1eIru5lcUJhpDiQxAiBLZstYfes0+E7/ERQkLTW1QW8RfuwxpgIRPsyq7gjPSYbyGxC1yKwybe1wTGytDCbKBznVblOqiytQStFEjfpv85mh7gd77DtcgP2K7DFKqX5ntFHpqoVm+WLQtojwrB3SV+OD4xN6vsuuza7QQ4nZ9qdcQ9gRGZatrKFC0eJKQhMGpkbWMvn05jv1SWkhILsqgyPS2sMXSGYICOX6WU9hud8BnxzeDZ5eEwaCq+3EwWcB2SHGvMRrSZOi1exoLsmnKK58DgQfceBmP4SDIAPE9BULgRu3dsPMYOVtWmNMASsRL2N2iiG1hHojU1wYmjpLRjvwl32fQGOfc5tEGwodzL8JDQCZjUoc6FhTLWWD8BVzZa36D5cbuasza2YS+9wOIa9B7YPRG2fwpa9FqsCaF7hxgcRR7kFgAFSzcqgIBGNW9xBgfygKpRqKP+rgCv6jMymmJUjrahpnaUVnAMlhARO1riZ4/7AGAVuzEP+9hG3EmkRtqR5ioHJdQPK66mQhINAdCyciu+6d2kDE+mRv7gJ1HDx6AFUCrCvVISnijCjq084Zdeb9Zoi3ZBXVHjoSHaXOQrtTQtD1yeltpHycQuhrBg1aTJgLAemVBBhDlKFLDSJaI7aoDxDNFPA4IBi8dQFoYBm40GMO+Z9j3Te6zWYxFakIFUK+B/rcFwVY60CcZ2W+k51VlZzJUvsIQyx0pRNulQXRAfGI5TEVSbOgwMtglT4oRNNqNI7xsH6l6GsLcYWRvbl4vrqtqb7bONGSroOwdhLcWpDYBHngoojxcDik25gCEYWKPHROZSIKSSyQ0kzeAq5x8aqHTT/NdMoYlpCkzYRXoJTsFGLnEHBb8Y2qk4bg6TYEqAYBg+tlmLZjo3cXIWjQCKpJK6qKSPkOwwWSfCq7SZyDUlrXtjfIplj05p4s5ug/e4aqKLGR/3rp2IqnJ64yfhk4nSWNZeXHYtYE44jxBA/oJ5hV/JyiHABcdFU+Q23lSU0VumJvUc5XIrkUNQT8X4eKsgcP55Q63AaD+GcuGrZgQIYkrSYo+ekd/RID8fxPS+et3CCb4LnJkJtTEg7FI+FAdArjHNI1pwp2I7oJcM3HuzYAgNdkIj/DBWoCnxStJAcDSDCEsP3Ly9WidMlnLoo0TEDSEcthQa9CD9xUWBoPC5KFMoSG37JK4xUTPi5hDqkS1D/JKJwCH41TL+k+ZIypp70zmWV3iS6XVKlq2RRpXSigdlYuqLdi6HSCZIzCWz4rOXB0IyWVEfHZVaQ091k6ZC5wd1xLY8YTS1OhUbW9+7Yno3iih6jU7siPo6Bvr7H3v56RfngrPr3LbkXhuYui3k8sKz3Bqhe9hy5DGZ0wpVnEqmBDEaiF7HTuuUa6pGkTSJ8YXfW7UwAwq/tHNSo8sJcUszpakokPiwCxb8moIE1nfppgtEbXXgqVxXJIR+H6oOpOsLZemO1YUFYVhEfhdXzVTq7yvHS0mRXZHXoLOqM39xrj+MZcgdYB83PmORRjbqRV5bzsgVsGFh7bQYcszRl16BpY7Q2snvYBGvNRQymIkJh8rBjkmm7rmOogcnEx3SQNHRab6+m1pxQCQTHgDSKzvO17XPQSqIJ/ud6bvX8yPjT39z+ZirT2J68Ewy0QW+odQAl49CMsSTb0Mg8jrBF7IOjjDHJV3Zjz4S+zlCOHbp9MLQ3+XZCIrHR8KR29ioiW97di4Z04+RaZM4JFtm+BIS9GbY7gceZdqeCIvICiGdM+I7gsYzvjBK32DtpgfcOoRwGggGcJ4Zjl61Ddr7K6ERMT7Cs74p4ITGw5KvzJQl4+UVmpGxTAtHPEYrKYOGS8rNb61CzbodXYw9De9ZysdzUV3wfb9yuylOHMn4iUL5hd4JbCQL774LcCmOkMtgTs1RCWWwgcBKhqP+EQ3/t0E6+S0CyKHDxztNOODFO9oPqP8wpntkDYjrEi/DI8Q8Y07BBeGTJaDrEW4+GpxLhSdmZj2f+uLsU5CkS254tnbfOpw9yz9ju115qz4ZSMsEMqVC0+cp9wVtb3aubu13Q6RcHBrc/Mw5zfERT+YOhXEpjtgFLhKfBIkQw+3cMZ3Twcn6vaiS0vg0jyzEYG+riqR7C1/QrYo5/l3xSEtkkX8BqkbmYOZt2MHyrz8auumvBqTNl16+g4cb64GlucbbDXIcCpiq3QskX32ql7y9C5aUCgx8rjvBKLMCuy9jLe6fRuDeIsEPKquE3Ie+6OuKrdTUP9XRsrtVhglDH9CYxHK1lsVwU93u9OlDJcnzWFc6uRur1Zfpam3wy8M3HFdoJZrX13AP16dCRGyT3oDj/3WT4DkLZExHWFZLTvkaOHDx4zG1/S9DYX37krIF5tWkUUlgERP4+nyDt+5SAstmCDX3m7Jb9GtQKkPDsRfDT6dEDCRRu/jvww6agO2++/XiR9HpGfcxABXPwcnRdwv5WNTvOXtCk9mxK2KLT+OZYz4QPms30wYkwfwEvBHykkcX8dqTanEJrqnwu6TwtGT5nt0IBxf6tMaJzdsz8Kh6MkFF9BTOMg7KW5Omd3lJLzsf5z6f/GaP0Ma/oGWPFKLbRbWVN8xVieYmSVweWASYZ7GjM8GdcEKlQNeNCLHYyMIoH9d6bfEo3uZ1dLeY7XeMJcHJp0+NwQEY9W+SuVc+u6IrkggHKU5Wl1cV4vYetzlnYTnv2NEx/Xl4XJNQojV6TQMGKB10ea1XyQFdE6Sb22NPE3bVBswCje1H/LbPGje81U3U9UbJUmkDcD7wwoBrY0bfiLaWf2jel6qnEMqzSIl/AdPi5Hd86eiydClZRHhoVlMXXQx91ZS80ahlaNAyunewrMyOru8c8ghGQFShcwYNc+oKitbRxZ1gYNV+ks+7QXzsWufGJRsZgdYqp0jJ1PVRfbRHP0FKayUN0Z+e2YZduLRM6FcugMc80ftzUukKinSAcJNDB+kXb6cT9ogoU9zMr1XJvxo659olv2iVFo6AN5Vw1kPxUAY8iTZ07D95/I0HFvXTYVGg5J7lKfPEE9lh7KnEmIe5A52m8FYQxY+luPHG7btZzcv6l6BlWxeCGRXkdqXpww00zi4d5XaLHDm1DMxRnB87hBUuAeg/xjqzDo7kG3sH00bIg1dcZMdbBNbgO4ly/2hSf68gUApYznQgnao5YRqd1cRiMnVMMbVW8yw63mOViUWdtNlqVrdjxGviu/lRcPQc66jytnksvy0fRAjavKzEmtn3TTdKzU6vorICTkfFIq+sBI/0p5gMb5jBH8W5ItEeR4FXbzFFdMjyfCdlRoZsq94RzcpR7QzSQzYIdknnsBMFYXaZdN5e8GepAvTkcjjApLN6GqCSI/08vjM9Zth6HlMsfxu2wtxz7w0zFbNKeCJlren1iXkK7hpRqsm9BUdEKdL9iRyv0GtcsBdRdplMbqRtWlpA9OfKcJHyO7jlwGQZvJKsSZoL56yO+hXMb98Y+2LY7uVBaXONO6awd/RxCmmS2uPQFU3DUBcd9Wi2k2JtVlGlLrSkZgiEjwuF0EFgvSIZJqI2yQt1yUx4ooks5nox2iE9IDfk9h+t0IzamRlQiVprhnYYVaqgbmSW1+K7pSRvCpvuuYGETY6OSeWraqdL+/GDK8KiKjS5do5zsreO9biPaCCfZVBKgNnarSJnjW6Rlq5q7SQMqEWPS03qbtAV81EIAqjKuFGwicYdbsjMQm9lUEANpSxGH+SKRGoJgXHMS0rbqCMdlYIXhSVUlHFKB3J9nFWjEOV/fcxFeYD86gYrPyQqa04qoU1jpAbqZzU3CUOykJo7eIeoIg7UWLm/lEGMGOmvC9cT0dalh9BavD9a60vUlLzZYKtAoziUJBGri1xlWNjaqYpnwxmbYtYpO4bqHvTMCrryzpkR1KkFXms/s0E0OKFMxwAMn2tQTINzdUio42AkIHhjbatBXmnGvTUi0VaVUkYMFx/w5ONc/MfebXVfTRCY22NdE3ozD5MN/VmOaBYaJgly8ldJRbY1qwj206tGpaNft+8MpCalXm/Wh1MilQNOakoFETMqZGK0fJ5CP5CDH7A1YkQoisSJrbEI4V7EVFJADcgJAmtGCfmuY+1ouMLSnu8d1QXcBy5dV6wqM0UROYNaRJmTMwM4XJ1T3zv3cMhc8uQNfF0Wm4rZ0xJZP6XRd7R4gZoCFL6LOyRrw+HsMRabfTrvRMnUbKGOGmvC2wpAY3l97buqHDmuzN95ei46xTLA8qk5ZfZU+/9PLqFsaE2szddbVrYCJhxcwI7D+Qj2OwgFKmRGce9xaZ7y3h1fZjSpytud47zxG21A5A7k6ZLdBNxpSu+oMfxUJStt5ZcyMa8AaD7Zt6dNsVTaun9nw6uqimFR1krKt5CSuVNZTO5Q83gHZhzxt4nrSZaTrKHJ2rtdPFvORqsFA7k7RVNnZbg1hrW52qh7MSh8TnjDIJdNJASjI7CkaLUVlWzShpcVesWnoYiRDbQkazmNQIQ56DBViwdQ0OouXFp+zcw++9c7J2EkiVPPiC6xsWd32xilToa4ePBwYpqnieF2EZH44rt/GAEP3O+hdckKXIt/FaytR2sqgZrVi44oTzM229W61jftimMb+SRk+9N5gpbFPYhje1kfJjIFhfKiyCrPPGRXLM+nBlb4leF2akCeacrDD1tR0gIzdLEBO+DMqb0ruKkVgjbelt8omoOSG7rVFV2ZJQqjKmqCBSGxx4inmRK6yvwabWt4NBemxD2E1tpm2IKgD1p6qZ7bb35FLQ0EqMasxSUEDh+LnYadWk3kJEvflHXROCj0ZBw8zi3nn1GU8w0Mcsj+Dj1ESCWI7UY3IWimJbRZI2pKp4FwKPWgNuTfNui+dWW2vtefGuYDF2thVVCTxl2o7e+ur2mN6cyZ2M8LcK+Y/MonCuuFrozD1T72YJSXUElnOMiSo38nruHc9F1KSPc0FQlL2sjJUDf0bK3yiU3Gx84MQrh/AX0xLDuUUUHAG55Ssjs7Fts1W0+CJZ1BdXbID0F9exxqGQJ0zZtPhKl1HbpHIuFPFcs8PjUyFId8hRPxI1VZBhzDp3dhLGDPt24su55sDut2Kns6pNvb2p1KjuD+21ptti4gaBXrQx4Pp4TqKfzcCa4yxii9GR0YHOzCJiev24qw6wf/xSKsD8NgKY4s4NkV86/g59paHU8fmvPAcuM0/4omjUbgwKWf5EcNJ1+kgyFAj1mOwZTIQXaGviKnU1u1USfaPeP4/mAVTLzByRPJ+63jKK1GkhbenIKfR6pRqfmweTx/Rz7f9hpAgudgsl5G5zfWSP8g/vaO0Pzw05ZBuZJ3e7vKLQ5TzTP4J2c16afysgQw6aMOrMgKHwsCw2HtqCHJpx7KQHbMAvQVC6NesKiPZD+OAyx/5eqvasUAzDahnoF4l6lhOQ7O+hrvnzvUo5DnbgtaDks0dUspssPjYcRytzLRUl/WiK7IWhmJmDGvwAzLOYSqwT+S5M8YWed6eclRQoESzCXd1sPLHBMc+nWzqf//dV2vVLpU1wBgbAlNJe+IsjVcHhBUabJHJOSb6kyxNF5kO5em2sE225Cq7QWdTeKequRw8e3lz7ykl95hkTO1ya8mHSFkPHGeZPWNZTzMa0ImDNM1WZOQ+21MOR16fZOt2o+ACf1ZcN9jCtO09sAy30sj0KXnaduMtxM29PeSCk8UMd4nuZzz09bH8Hr7yLHpTct5LpLzunn0quSpxf55IB/p9Pzu2LpBRh919vEi+AZiBICv8YmXJ9KaveOD9pkCUrrP3MQfJrQdKK6bK5eV4Nxi/6VDaQqbqi+pHPfmGj3/W79ZwfsILuTe588G/f3rXDnCPvxckm/Y+7BxTu7fwnRgOWiT9OqJUFM/lfc8P3W37VUPvxuDZtz7fNfaBCaeLDIjA9zg7TFtRiLd8ZyQj2YaZWlU904cfceKMtR6XklZmRSbl7AfZedvx0rSF1Nqcbwm2RPnIJfGlehjuIvokes02tDxiR8ridG6UrGE690qeiM/2FnB7AL0ZSeuRA53iJVd2ODWMbsbUeuNq4ERBfnKVE2KHiOmW574f1JzquJx1uY4sSzem8681deeXM148t4cx01zolG7mtxi/PelJbPnq5JaH3MDWCP7MkS69mOR8oTZVk7Gv0/4QiWpG4opnxg2qdurRKLtuz7h2QPTo1kHUD52ENyuvnlTh31sUpewYKVOq4dAobX88kuBLVLznrnXF6g5f8WW5rUFjcwDKVsFKDa/kpxzpiw5Dd0bF+H3+CQzGfC41TZXu5fZSzqb93UtFma6AMK/qPfR6kEzdEA6Jr0A7MaGMkaxOsnR2lagf8CT7fuRWa0XGwVg6Qdqe8huGo2dEqRuodwC3VanCO6xf3Nzrxp3ZN+A2P/pLMCyQolKET/2IsHsBrCKDPPfALEt4+RGCtDScgeTqDM2oLxJF3ooN5OACLkrKak5VCDx30qq1BBdgwSN+Mgoe6cPo8/bcm5n98gs0eOww7srPzTt0PNKts7lxG35+F2qS8QdMSaM4EDor4Yd7I95kIM+MuBMzdESbvuaoRYplBRcJ/QgkznPb/KXtGs60s9vQBKN+lHQkv0jqoTSfjJl1Oo3ylfyyuYlIp5WlchKp/ODNujAuPD0xIsa8Ohrdk60hZVHkBtreHOppz75QVzh4uRFyoTu653DCSHRU4mhbxKJVowJMNSQC/hoMEgGdYWHfhUF7WRDWq/Iz2uYZaNv2HsmFbN4xjTog/aVX6q1HILP2ckY+RyaWkzxAVceI97c5AAfBNsfdwAXJvzIORHp9dPjj5PnF/stv8QeLjF/NE6Lv6zsm0qmeGkoMB/1t+1grba69CeglUPnMOf8uuBOJ5RDaFA9Kct7tVIXVKN8bOr4zWWS7OVN+//1bLuJLcXZYl44qvrFxLJJ3a/VU45u/gOu2KB6j3BX5VPw+eJfg5LBHukiRx0fFcdy7I6PCRx5t1b+beILK03Az/G4ehXvfQLChRAfzFYtiAnnoZqtkFZ7gsZBPtsAU2VbdFa3o8ZTy7XTU1Xq0pAH90PT2L3CYVCyUZJ7j5dbFRgr/Um1mTxFhk7mte3JKpqbvZoGc3+4V6RH1Sjp3pLw/qujRt+nmdffWaAFPuuC25r0qrae9L8VwW3vv9Nsezp3AlgOJztFzyg0+6BRy5f4DnkzDi6mjyDpyficH5tc6Lx/ruOw4Lf8wafJoV+Tv6Yb8jblw8d4f6wP84xxyX+GMc/xwegbKIfnXQFUTQwzBTMqqikvqp3CQy0B3/5heXuJvbYgTw/NzOBJQqXa2ylJKqxokcHUJnIXJP/CJHg2PMU1knc4ynYdXU9SdbnBYXcKpp8AIPHiDwZmzKl+jHBknybycJUls9KSruVS6ROH+vkRPD1SxqbHEUz+leBlD6vb0x2b7mIXTQqDkwm19+Hjm9BI3UE0Dbx9TzN19LM7ARltA98mUGbszFvWL7uQf7FRtiv18/pgeq/QmX21W+1ew42uFKRlKLZCD4cF2VJtyvc/WwT5YjRtOPvFBeqFB6TLvBNHkL2E5TBKI7J+pxvecFdk2btPl8DqWYiTpW4Tvh2ZaAAYm4jOVmOXXyJ30MjOTixm7BWHB3RZpZWWI+lvv8sONXd7a9puNnql0Dmd6Su4b9vqR5TK2Isu5taqpgX4heiL1QdzaEt3W+qnbo3fKaiOQqFU8rieO9aeXOYm5SzjvZJX5q5UvUMQQn8IsjCBxStezcXsgJLwNEGaSdFMqDaBG0uPAnOzAxAhW44VyIytl4aO17NCEdig6L41f16JsMrWEdsMnwYuXB7Ddgn1ZBNikyoSXTQptXkILc80E4o7M2N33elmWGNbcXKVF4MqY2AiHRr8EnA2M6Zf1MCu+5BUsFTk5P77/ODl6fzxJziZnZ+9PjpO3k8O39GDy8eTN30NLVXcA2nPQ5MIISv4yADlzEwERBnI67IAAEcT6Hf9Bu8oIINdYdwyAb2SwboSr4QZyc39VmRqHhwbMdAM9oAQi8098brEggrTGVmbwLp2bmwqmOYs7BTz+/Ke9tpWRC3PX8cKoSYhLysv5/cck13ORqKO1aqdzd8/l1D3tOw+2v1PV9rUPjNNz9Us1AuSe73yKZvzcrhd8AJonx1pIlBCaUOB6kqAeShKJVuddcHaLQaKTm7yJWEvFe/8PBah9pA==\", \"make_report.py\": \"eNq1PX+P2zay/++n0Anog9RqvdmmLe6Mc/HSNOkLrm2CtL17D4YhaG3aq4ssqZK8m23efvc3P0hqKFFeb9IXtIksksPhcDgznBlSYRi+VVeHvNgEP+a76+6H734K9qpr8nWbBOp9XWR5mV3lRd7dBV12VSh4nZWbICuKYJvvDo1qg21T7YOs6fJttu7aWRiGZ2f0Lk23hw6qpGmQ7+uq6aBpWXVZl1dle3Zm3jW7OmtaZX7v1ubp321Vmuei2u3ycmd+Vq15AhS7bdXsze9O7ettXlhwXb63z4dDvmHU1lXZqfddkV8Z1PSbfVZmO9VwrTrrrkWVN/CTC7q7GpAx75+Vd0nwHEiCBEqCV51qsq5qkuCnrMZ63ObQFABrRmM1LeEdj91gWB729V2QtUFZ2wECveEF/Fdv7Lv20OUFw23fFSprypmeNgM6OgvgT7ZeH5psfZe266pRCb+7Afx2Kq0btc5bmAtZeJUVWblWm9TXcl1kbZtv8zVNYdoo7ImLtpeyIsxWWlRty7/2Wdddq9s2hQrNulJbft33Dw9AvXRLENL2UPdwvUjq+vJNtU6zw9q8ijUHrq/V+l1d5WVnyPLL0186rBLADO3zdYo8lm6A6knQXmdffv1NSsxDrW/yP0yznSpxUhWUlhlgyqx/dnb24+sffnjxNlgYBp3tVPcjPKomCvfZO6WJFMZnz/QKQUa5ytbvoI3hmeUSWQsw6JpVEvxclWoFoDdqCyPNNoRjhLw4JxaMg/NvkefmNPLbvLsmRp1VtSojVa6rDeCxCA/d9vyvYYyMcw0sVCiuz/SDZVnS+poVVbaJuEKsO8VXGm+YshIGGzUHoFLeCAQ2+bpbAsIJorJi2OusxvW+gaHpBsFFEDKIEB8doDPsP6SGbXVo1gqaWQj51j7P1Pu87dooDlQBSwcxiFKapjSNZzANVXGjohhXloKJ9vUoutJj7wnLfZuxV81GQZ8psXpaZnvVRntexnOznnnYwFQrokQB2OErTQRqA0NZ4kMAooneJAEIwRIG2nRqYyDOcpBWMLAkeKfuFkW2v9pkAb6bI/QIn5aXqzheEWAgiW6OhTdZcVAxwadHhG7A0guAGwd/WRB6UZOVOxUVwCCEXhzHghuyHMj6T2zzomkqYFxgSlWkBhpSKXj1fRvsD20XXCkSlbAAqoOW/X+opgIGF9SlTjRFQeLn2zvBvwnMbKd2VXM3D4iSa70i5sFojfwvrQYiMz7MDSFsEzsK84Y66rvAeaW5DL7PQWx0vzx9eyi5FSKXpnmZd2katarYJiCPc8Jp0CEJIpTTyNlGZkdQObbFgBPXmLUgdGDigfJh+zQMcP6rzhSWqiuq9eAlrV7oNa+j8CIUM+OdnW34qoQJzjcgy3CZBb+9fTUPPgA292GPD45ndnVYv1Md4Oz07tYBAbvN3/d1BsjYyl1z5yKmJeNV1VVPbYF6v1Z1F7yiMsIX5Q+89Q0KZgK1s2Y6AhTkLfDQ74ccZQDydgZr5un84oJGuqEphFkFsUacB4CFUNuBmoCBVO1MlTd5A8INhHEUPvvXL+nbFz+8ev0zNAOQvvLvX7x89tuPv9p6Lo3WRY6SZcGD1T8jnN5Ed0uSYqFRgD6Ia88sm22q25IELbPZRrUd6BFUokKiDvlth1UA1kIigSintgyWKqzutLr6N9ClTW++FJhvq0OJ/PpEvGkQLIkKC2Kmn1T0HXHLQnBOErwh7lhsww+CWe5HTIqQUVppyIoJ+xxNqrJrgUrL1aAF/gGhh4PrGpZ04T/UXbiKR9UaBSZefoPqAVosUYgJZOL5alZ4+FUsTFxqFghgap6BDTYt6k/PsrNiBWVdeVCjQlhIWQEoibkEfWNA+6trFTXbvwNOjvhHu/i1OSg0tmkm39HP8TAkCxhuIh0YOfMFBCIzIqL+4jEc5oovFsHl2YBCVOJbpi+hl5+r7iWWGxH0cxVoptMQQWFUt7xUPwiM7i+GjNMvikNNg2BDTK+MSS3hWSFsijETDRn0g2kMjyTRcH3eh700Uyihsuaub2/B3c+6fX3+ATcLM/zrK7AtrtV70ZrMyvawR+7tjUbSPEJy5H8oEqskT7MOoMAE49tpmSpnWZOH57hrGHoSONPtDCMJXrzvmuxZs2sXH8KfVJdtsi4LQTeEjCU8Gszv713WsIAG0uYazSSeaa98gCW7cJBwwQJroa1iayyNTPhRlbvuGlY76kmiFKxLW43Fhx1BEny4j/mdHgg1M2MZL1zm21evtWb51Q4ONabSu062NIJtBgTeWGAXhAzskOw2ZyBTJHXWVX1nqDNCYopcls+SUZPnAO8XMkdhArk9zJkDgYTk3J34e9hpalqxkQMCaBE+f/3mf0K3D3ckhMhHzLcdgHeuqfT4PFOVP3mOX9JgPnF+CbFisCZHi3QkjlWhYGf48atEG00v6B+0CUbd8S5zdgv7fLDIUbMeio3Wa/sKtFq/fvX4Uax9hpp30O/Z2X8OPB0oimGHDhQBo/IP2LAeyn67ZySv8WksOxBLSm9XJ6z11cqa6YQhw0IZ2HRa3ZKekCrXaFPa3OkWYm831FWsTc2+8DSlxRrRVr3LFZCQXvKWe7A71tuZfdWhFBebB4tev/02PqeZlTTfG0s1Yo20AENtd93trvbnPEHnvDO3szNBClse02b2UIZuPW1KxAJ5xNiaCEdHres6eti6AKBG2j5N7bbK8QCAuVKi42IzB5O4KvTEJ9Z059fsqPAzyVzTN9vckV1dFdHQLv/lafrdb8//8eJXJBWYGJ7yN29fvHz136G2c9prXBYpo4Y+CIIOLKORxc0F9s1+BP1SsqoDYOQusUyCZCEPg+6D/QiP80uAos5LdkvES9xJrFzQy1DjF66gF7QJh+V62gzJqZ75wXXR0QUvtcsrsm2ToLcxcVrzjTbD+8FMuG/grfTdaIHNMHAvpJc6Glux4xDQvo16ZNgBYi4D6v07lyBAEFsR95CgmeY37GLjv1lX+6u8VJZvW1BITdtNuhbQoIIhbY77Ho4yse0KnT60C5LbIe7fdBOTmoSileQ7C8HPc9Zi1oPbRI8ykwUyno4IDSge+00EGqZjTWToCFTnNlebNC83+Vq10V3aAU/Ng7KelZusaTIwR8vDnv1oqiWHVgJ65n2+P+z1r1ahpIBHwrpvadUHbvUYcBz8fWFbD4kELbPewaUbwC67u6vVAgqhh2++YnZcgz7ocKLgNYyIfkbYvqWedeNRW8A8LwuyZRZiVJrFs1v0XDLkC/0wA1sjioPPDdJU8/dD1WW6921RgVKCtjF0jr1FLqZcdxlpuN8GT+LgP4LIgIB9fYwrnrdxt9cgdXQT3fO3Y3KtQYrmYG0pi0LWlVWJ7jsD99vgslcWGoW+2ZIovQPA0ahsFZwjJeSbeAUvj6D4949C8e+awqfhOUAJkBxjjnjaHXFT7rhr4KhNtQfrbpsdii6F9xEyrFY1YPOtO/IHLnkp4zJjpzEsCfUeVxvzpOQX4d6p2pziYONxMg/iFAt4sdh8Vq1CPxdgNIMfsPwiCy0hI3sB3EpOYjPWHs4KeBlXiW0Rx6i26yKDTcfLDBSj4/uiQc6yulblBhkUHdARYyCcC2ABBwIFLreluzVslAoEFRmd0AJFeeAgFFHkkAPK9CdrcY+x67rHAiPvQWd1FMvCgAR64mwkMrIKrrdbOMZVVS3Ys3MK3J3x9he25luAhFGd4XvyhreOaNPoiGDGIDiAMZHE0Q6TCgboQIrorA8mIKommtBzD1TEOdXYz7BkXwHp01o1aU4GOow/0vOit2Om9vrQoBXi1MMN1uWTJ8ciAW8sXfsIr7EytAOGcWjQ0E4BWmiFIhkQTiQIXrL9qmOV4xq6IJQwTnWXCaCnNsGFlpKsZ24Eil1+Q1pZkP3vXJSX2yoydeIZCBe2I/nV0y8JIK/dFHUpDkwTAQbGBbOyvuOx4VrKwDwUDZJAKCLBeUYb9cjCkgUeqm7TOl+/K5x1ewdLobryIoAFPQL4iyP7uRG2GJ7GUHrWUSwx3av9Pqt7z4aADQoRpn0R3n4RCmUJOi0DSlAoFd6QOhYDiR2TgPHVavn6UL4DHro1TO6srWUolndf1fiKUfLS5rKXuU9YyLl99y2FHAY7s4Y+UWAyjC9EvTGUXub1KPH+hVaZfhk58mSWw25qSdDn2N2KyWBX4gJWjYQr5kW24jky/NF3P6b/BDAoPrTX0UBsWzjT8prrCUjMwRzXcxiYVngf0IvdioJ7RhVZZrExybuRoYVp8fOvEpe5Eh8X6XyWFJZu2mb7ulDIQ96qqOkNf/HfujHg5plcjeOKPIKR02AZMoLhYAIl6n17bQqKpS5heZa1KeHx4F7z94PqQtlq1lWpfh9JQDhwsAhYeiRo4wMztJhrsQj/aLuNluQ6HWNgM49Gh71Q9kr0GNt5n5X5FigB0D/YCQ45JB/Og/BaFZvz6tAR0YO2Bv4TPs2QRg1KF6tuFCwo6AqEfL4WDBQwaZDTW9XcYN5OiXOXFWxfIWvjANF2cWAjC8yPMIeoq+1YFkvzR/BeDwI06yErDASUO7qJrMSGHNu/6BnmqQHSU5Bf1tT8yXXJYSyZluTz8nIlGhxaWGsgSrU+N4p4HjB7UMV7Z84e5MbU1BTpF8O0m8jUSVy4juhwufaYkHFgTFRFcaYhJkbknI0EH1uKqlxf77MGk3Xsc6qlvbGtEpe4STCYetvQCJQekqagNIcukJProrrbo7lmqx4joa2UDEA7pHGLkt7WGtJHm9lLR7I70tudjwHN3X5MItMk9djaDlzjO5g0qidzj0yeoHazonO12F3JIp0op2c269bXqY7Rme2SnTFbGJr9ksFPWwC3WbM/1MZaMe34ba/ZrYmyV1kLrXFKh21EkachmKICU7BEn2AAxfQO9vvX9MKBT2+PWvXfWbbGWECDYRJt07dBpndyN4p7Jtereg91ijvsDfs+x84vsBenZ60wtg0mPi0CV0XO+3H0G+a0N9n0mIRpRnoS4SA8j3qx1iYq3UHYfGiQCY/UhN2FUoFqmbRD4HcQDG26hxWtwdBzB1tqkI/k7Vsl+n9jUmMLDPpSfubsDb/QBkGtMjAruZwLZmBhV81dSluLeAZlPspIEgvyMB5stC4ozXUGG8EtS3wlYwJiMKdU/xS6G0KdiNa0Bf3whCnKbjlx3PvWODEiF8XzEXXQaQcdPRkPSgDBzs/d0Y7bGU4ZtBIT5+mrZxJ0X5mfyTTLeFg4GZr1uktyQZN5fdhHBjuMLCEOmkfB8Cq7nOIn2mnP2YRJ8Ps8oKnHTXBfL7LFWjRqnwTIfiNeUX6a+eVSYCpM2Sl30XiCZ5TCqiKTwMr4ffnV559/KdWTMBeBNF0FmptSHcYyeG7kJek8r7SdO3LsXphFQhXMhSiWNll9SLtrDJ0Y08/ycgYrqOUYCY7T1EqCJ7FjPRJNEGy6v0JcXAqKml3acyuNtv76CUwheYAIOXgppmYgxb5+AsosrP/29elN/vZ1fD/sHxnrtM6tzDyxZ1t/1C0x6wmdGqY+rcu+9qDDAl2Sa7B5+s7YpD3SG7Cplz8srB6V47AAlylYwEHVYXddHzqziWB4tLAdBsXIgzQKLlwZIGAaEcO8Z+WPXXRyKNq4Sm9Ug/tE3Kjsrmap+Z2mcv/E5paoa04q+OuTlhG1Qcr4K8KCQ0T1uY+ZZlewayi7077fZ+vrvFTOrohpAKsyvboDo12vV62tb/KGtmAsYUG4Uu3YbH20v/mwP3CKn/A5+8NfbDI/FBLrvU19KIyTLJx29lmbu3+m01IPBA99FAXDVPu6u/OF04RTAi3YQ8vqG5pg7KKNIscJI/f7/oDctPPOdvyxfjsNYKTHKWrjxPxwnqSrLTbRI9jWt4tLG6ETgx/RbeCqs2WO22qzmWVs+hPhkmAce3QdhSNfioULz5deV57uUqtJB8ekn7M+XA7CRNHZmbr5BDbWrghZTbAyRTUoJhKIR3cbpxkbT/IYk6Ju7CNlvMPkrs1j3Zin3kHV2+OOXf7I0Jze917lJedDjucoHgbn3BDuX+NBmO/GBm8c2BwGdc8Q9PX1Xs9507O1BjHMfTLEIw60j056E65/t5Xe26+9y2KeyJGujnmbhxiwtegczXKxt26F9SBTWeDOQCbOrJ0IzjKPscQtlr56dWOqGTR8tYzvTVfFydWv3HCr2xbXqUHybESz9gh+di1MI8fABUGmXfq3ClV56864M7DhRBuBPVwdqCx+B7U5qw64+xqvlUTkZ8gFN1o0swZmuYicOJaXJz086LSFZbVn5MQmAUkKJkFTpdVNE4qtDJhJZWQ5JKYMHfuTtSXHZx1QTEAQqwNomlEjIa80rRf632EP6F/RRTovYqrPfQ7o285OXFfS0muYAhOjrxuJWt148QAYZujTw0bZfGTUdXPioBFhZ8yPFgPSbz1aHom7FD15Cpxv4NdImsu0Ik3ZyYhnZ1M+KP2I9ANx2HDeHyf8U7MMdHpBr3Lrzez7rMteopNHq95hlMkN3J8YbULNs5lRJqEv6MSU5bgXVDUhqrqqIxNKiv2hJKNpeHds4hvmtCUFQMzGWxYaBdtvbPPSTdz05zo69XW6I1sTvp6d2ksRfSkO+xJ1QUoHS41bt0+kPAGTUdaltb7pFPJGR2wcfEQbcj94iBYm7lD6vBEfhTEX398jup/9DQgTM8OaFEfPnX5nAiqDISfaCe1jPjMIPrkLDL7dAp2dsKm5wMBhabcs9NQ/JYmEBQezvdBTQ0bshUMUYm2gvev0mtK2WJmdH9AB/mBpKcp0rrNcztEHw4AoOOWsJAF3P6fG90M4SypNtVfCRKxvrxWIWIHJtwFs0WjQFxJBcmWib2uMXf+D8rm0DLW0yFroj06tcyYLJk+rji2oaNNUtfAyS2zFZnyE+PSoZtDMR8UliDnYQBJOJljPZsslbyj7qjFsNy/17FNQ+vTp1zHs6fkXm2Oqa6effon516WPZQBuZuLC9Ot+BHCpq3lZQaJFvMAkuHDwdbnBxVX8cvlB4vY4tnBwp3lkQnsnUlSWM6lD8L0i008ifxU7nB85v3ed1SkfDLQeERE91IJ2arH3+4HbNm0VJXk+cUoeypOi/scZUjIV3O9scYc+5XR5hPNlANBzqje7TSn6mF8dTCLsMAjkPZHrQvYkV9V0fwOD1lJ7FGUYQfYeHAYTs2x53Y7QZU/GkVPDcmCY7dfhKRpvbfyzdPdho6sdRv3DQNk75QU5Hs54238EWYPJqFcvmZwaM1ixe/RVfHlab25jWOGYmkKLxu2Y2UpYym5gXr/j1Twe+ySWT8m0cMs4O+bJynhcZKcnk7CD9dXWVauiAdugEIJV92V8Ai11ns58heZUdDIB/Eh6bpL4rYRlxInsoEm7nBOifvmvZ284hXMefPAgdO85Zm8l3xfMQFetO270I+F/55ermHQYMS+Kn0ufcLAS8As9BQ4vgHIZ0mIEAnd7nkXr/Bw1Gm0BzR/cJqcwKD7xzSO9oAitRTUZct4VGPbvhjdj/ETBBaL/fHh81iM30Y+PwdmJ86+u2lCbw7pPmPRnUfLOazn39LXyLTEcIiLglerBxQXw8bEESpciIwXrk+WDYayOaX17FtLsJ/XwhjDOjh2mfcuV7QZW5wt2VfDZhjgxyLagO3hZiPkLj2i6idn8y8LJ7BPHQTDO1XJoYkAUkVLq1HQSS3nc40qfmPvpTfd7yC3gS/sb5Hr2W15P0p+su3TyIlc6mWBAVl+rQ1ftgffXKbGCvq4sQ3wQSDiayz775VEJio9LUnxkoiJJGmP9GwkkNwBU/lj73wBKsTlnPtDv+yFQbU76twA9arQDsPLxQmI92ANIdPsf7g7Axe6RewCBOm8BEIZ/B2Cryg3AVdaCqVDK0K5/Y2ezlBxG9KUrDUOJbj/bS4xw6KvzInPcw4NGolf4YnCblxTWeJJLO0gXITt6E7qbC5brDXlLF0+MY7MGppg6a6Ga/aGzN/xhReu2OnI27iEdsFF4VIsChr0TknydtAda4TR9wIHMg+XKXpuGpq/DwPc2oqdfJ673CRv8kdfRgOs9Hiohfasm3+nLMNwZXTrtRI67J/VtkLWnSTewxY6C12f6xAREBrNheMwbyP6TuNOadGs+Pz7iT9v9n8uVveWmOWVp6a5DXhwEkKvnnLH0RNnGAfiHaG9o7cTTRi9HJqJcL35hPMrKH0llkQuG8s9QAOot3UhNTxtsCGtreqXI5H7Q6H6gUDCEmQSbTbUFZqCIjREV3waXHKJ5MnvyYJ/3sUeo2/4fJ9RH5NWyXbz3i/hRQynpOUTjRgjH3rpUOEsZRnK0Oruc5tIZ5SRa9Qj1jTgLzMVUzhwqqnFX2IlVYTLI1d83ybwIM8RjNXdNyhvmyJ4bOMkv8JIFulRhtm5vxI1ZpKtBAMLbyGNCEkulfHxwEX42+2YbjowebesMXO9DsweN0azJW1K+wj1rwyoh2UvWVbuCpdHAvPe5QcIXOG6k/YLA52jsmlI+6aUhjXluDMbhvwGweIRRb5l4ECIrxYuPPgYLlXT4BJNcxNjdMflQczsZ0HcZdlV9+QQP+1xXG+13xcUUiTqy+xXvR8UO/jKegAlGQKvK9sBW+4N9frsIng4gpVMM6lt6fbOebcU7zbcD4KeysGbfUeuHONk5aTTUxkesvaPhaF7OOuxs7yVm02t4S8+ph8+9R88d7/aR4zUn3Pnpu+/T3jcur/yETXej7yMn/Xy+Uaj58f6cwcVe7fAOUMf4pAuZp24xlgFU/J1Mn0fXYdhk8oz6Rxw674ObhsN7VHzx3ocixgIWd3Cd4+05d+4+2z1ipqvIgLK+7tdp5SI2vhbYxKJ7e4/sqMkrlGX0nyPiweiCYtd4HBwpGMyOvd67JT+E90KCWfdeZyho+xi73F3NdLA50nZgXqgF3oHU9/WnXqQwiVqwqYBqdAtQVXbofzKHraBCIM4oyPQrfV4fL/jyuGPEXQPA2kD6lA/rN6HMrXoIhr0uwAsDqKK74UxkHADedmRyjRObYkxZ2Roc1/2LuQAHU7eGRPsVamia6bGaa6cJ7gXBJKe9HoYp1vAkftQnuayt/9wmpY1Y7eRrMGCr1cIets12jVJ8AdzkyrgjqZ44udJHU8hNTpLPF/uIixFi9/L8RF+ZDzruMhEnuh+4gj8a5MQmdjxm3ydW8mhXJ00YfwoVpa/ZZD1vLvIxemgs7T6BJc6R7ReV4y2oPRgMiDMYcZ56eCFTQulATCm01y1ZRX4e0Q8KDaHD7SVu8y4Tzt2DQWGhoANmt/HbniJ69+QOaOKUcF+uC3rDxxZZe92B9UizZ9j4yMFh4k0/ut5vR/SyWbSd3TYg9lK8ozLytnokW3ZZg5dm03QvnKnfwN4eNPVXE6yLtxwOvqYg6SJQPkKUfV5WDWcf5fqusWaHkfPel6qXI6dSmOzbwekN4EI8vWE04iceIc+3AwieuzTl2ffeIvAdMHdDy30NcwDYkxJ4em5hPEOHQKQ3IIs+6hF7MT1ySn+M2MNH9T/mvP3HnLk3u4UxjtOOJ7pCEphTnOtkvw5uzviJ71W3RoSpiffxymsBtTWoeYtuMzBfk3E8JOabMH0+rvORmMhdjk7u8ei7MhbGxBdnjgHjPG4pkhmUPy34AfHwoHNy3LMV9joTW3yO5v+tT1Yn1J91yP75fdm09k/vzqaJPzC69dr2NPpQ0FEm0II19Wj1pSlcTTZIe7Uvz27YhtNdubTxtzAfQLIVzYuxWTMmojNMtIaGZxeE3bQcHGxYeZoOziqMWjvlPgAmCX/cLxWs3JR9rGXOGTgt7PkDTwNxqGDYxhb5+skn+hkhxhswKwqhDcYmneOnMDNKDQ9x2/fRiWe2B5f9ew/SWtGN/hArlecjkS6boFQ2NwfNnROSfSWf2sdjwVZhCU+xFPMTVoSscuwCGlkvGQN2rKZR6RGt6D3I86xXIBLWstdOKwD5ndYqwWT1sUaS7PITMip+/sSqGLf1UAVhn9zmrVEOvgZac4x7enk50QKEDcL+l14EvopSaDuQnz8fAwVxS7gas/S5lp6Daq5wdaA6Lb2Ij8Ql9vhjtQt+rDx9WVkpe3n22/Pzt6+fB0SE89f/fDue7pFg9LU3tDkKYiAARe9avEz0PBIyb96eQ9PASD+3mSMasRddW4g+bwOv/DM95cd6yk1PjxF+Lqi+BOEMpd5g3p3S1SfKO3HVhbmHwVzhYOXZ0r0RY7UcXw+xmoTJ9zGcAnN4icTKdyXGQ0jStRknYcjQjqNnoR3HjS88OIYZ36zxMF4a0hGsekjHcBpcrWGvwhCgpm/fWE3fq3EU0OjqjdXxSzX6CzXEAI/evbGavFFDgJAFuJxgf68aheEr95qZvoW8iWzyzhlRXxaspLbXqnRC0ZvSsSdJetKWutoqNn4lCfWRXiW3qdcEEJe3oaYGtD2nLLUnIQnsPt9x8Tgb/KF9AcZSdzxd1ljrJzlj+1sg0YGZEvTT7hl1jlZju+GxZ7oZQqI7uBvCHI401/R5QNBp/ikQzln+PqgELAsNOrxhc+VaYYOx9obp4+niHnLugaV4JaU2dSXe8cQFoNtDUZBGCk36rdPKvT9BHzh+6NbQj74QVK8vGEKKeKWE15zOD8llOTKle9ongTe4Z5yPblkiQpUuj5uvNC8mvmHbO1PtShJ8mEiOEmGMpPcxJ2KJ9p0L5784d+0BKecoGcdLksDdMB8NjS+HrmrHRevZebhCaETTzzWNzCWWCt30PQXJ3vn/vmX+E++Pf3RUnO7Tx8TCYzfrj4Xu4ApU5/ryozLYTN3numeg+kRSg2hpJoS+WQp47fDrwDB68y3x2c8opetsrf3S9BIDv7bCs2Z3wH3yGyqJNqpdNzl9fWuRphuQQ2ksWuLNPdgNNYnC83NA6RxQCvtPHul4/7Uq6gXsdfAbTs4nSzEKSp+94o+CXfB3oS70t3OOdsbfxznvqnP64mhGs7GwQ/muqgqVla8J+6x4Zm4F5+TcBX+K9Aj49qnpwX5L6NGdiOwGfe8Q9yUnSM/ZHkz+aPCdGvPx7KuszdfPmS0LdaOKhSl59fPL10lgzYkoa9e4l4jb4DOuSd8cwl/8MIcnYIE228EPTV5Ew3zo1uBkv+E1/AQals/M0sCElp4NeU7tJ4ZEzER/n4ocMOv+O9+0TEEnSZi+76GxThp/e8s95mPQIGj620l4oeJT/QpaDr4NdTY+FmjW2EYqhtF6G31MaTD0xDtikY2rz9pQICv8wfaJB2sG6TzBocR0EPpmHX/itYsskvSREiOmzs5y/HAzznOaolYP0xSZKk3Duc5kQQ47+z/vQHBT\", \"model.py\": \"eNrFPGuT2zaS3/UrsLxKmUw4ssZJXIk2Sq3jdVK+cxyX7dyHU+lYlAhJXFOkQpDz8Oz89+1uvElIM7f5cKldj0QA3Y1+N9BUFEVvyt2+++WnX9nf8y4XvGObphZd22+6sqlTtoZnVVlzdszb/MA73oqUwd+23KQsrwu2yatqnW8+iWkURZPJtm0OLMu2fde3PMtYeTg2bQcz66bLEaSYTNSz3UZ/+odoav25ana7st7pr4e82+vPLdefxK3QH7vywCXWAjawqXIhuNBozSMzg+N8Z5i+pwTlc1MrSEdAWpVrPe0d0kAD3e0RaNPPX9S3KXsJ+8/XFcD4NT/iaMo+8D96Xm+42WndH463LBesPupHR+AdPID/HQsJW3yqeN7WU8lcs4XtZSY2DWxdztrs+ebTsSnrTk/Y5HVTlyCGbJ+L/WQyefPbL7+8es8WmpXTHe/ewEfexllWgxCzLJl8ePfm9cfs7YtfX32AmXHUtXlZRymLrvKqLEhS+K3joouSyeu3H1+9f/viTfbytze///pWLslEfjhWPNuW8E9Z4HT9qG2u9RNgDq8AxGRCgmAfERFQ9S7vBX+PnBIdL+L3fY0yeNW2TZvMJwz+A316n5eCF6ypK2DfFpSP5azoW+S3y4lWggF2sppfs//KdzuYAHogYBtSLycF37Ks5XmRobbFKOI5STZhFz+iKCXS67Lbk/ynzZHXMYixKYDaRdR324vvogQltgfZVVzOx/9aDspekxZPqyYvYjkhUVgVR3nWqa1nYGLbchfLP3OtOEswuxQpWRFJb0EdJY5yy2CXavoyAm3K1k0jOmBzXxcRTP/Lgl3OZg5FyDf233nVS47G0U/ajger2aEXHVtzxm/yTQdsBjggLYUWJlYGL2hndZuJriFqAe05fK9wMtOTWSnYtmnXZVHwGj+xbs+Nb7H4NKqCX5UbHq1wY9Hm2EfnUH10QMnttH0NOsNevvt9DLo8rPMqB+vMSEq0E0IDVsQjBqTpmVuekxMDyJy8oTvxUbw2uJjGRS5TAWYGsBJCA6oHkniCCJ5YysFz+oIAw8nA60h9QlsTDwhjzCEAwfgVb28ZLAftInkQPCaOVdkp7OT0BVi7Rn1oCl5l8nG0mkjt/6MvW15kpEEw985QEpGiocaD4wSesWi3LrootROa9T+QBVc0eOirriQv4U4hr4gwWrAhmDabzmaXzjgqNMy54gIGv3ZHDvlNVvBjt4eBC28A2IahIYO/sHQL489mLsr8sC7yrLqU6AIjz2hkCHOH8uiaTLJwuBbJWYOTBWTffus813q2bXOpZnN26S1c5+THHzOB/4FonSFpSYb/aEveKPhUoLwEKW1g/GPbc2d4D8+bHcg6O4L+ZaL8jEAun33nYUd1/KPPwX9/Bi2A6cUYEgoJR+REZANK6/K5M0WOKJ3m4Me1aCyoe/r3UArICsD9C0/ZPvHbObuL+M0RNIojDfpjinomeHtFT6XyYlCMYUlybwCgW4InqVmHduFr97Ts+EHEiVkD5jmAhy5CA3CIhomW7jO2urXew2RcaH0dyp5dlU0FVlDARi20+4G/dgiKYE8g/Q0I7xoQRUnCFotTszBoq1lnPbsKFOCkWLNlPgbp3zxwJsJAUum4eoxoLg2oId0eAzRYP5sBpT8s2NmQ5vqiqbPcIMzZsRElehcMBUwOAzMh1VB0oMgFB+liWoTijiP8iomLtin9fWilZoAcSQv7bg7ymcM92KcFj34cUCifCpjpgQBT6DBGKG4szYJVikxyoJ1QGI8Pd2b5veVDjYA4JIB22yoUgwFF5HGza46FAG5IQC7JweLdZ6XI+loFM5oj8zw1Qar4seWC1xgBBOSlkNMZNFNE3woZ7NQ+E6MJat1Zo3itI+mFiaS2IGE5BFODDGxDQdSGoTNqIIz7SidHYEPLlaVHT/8LModiUgaZdNVgWFJBKuNIV3R/Tj1fUsgElSkMxGGu5QEn03GhJ6Hkz2a90QqSGeQr5JcylxMmFTxHl4UwHSwP5IKRTmL5disDdeaq2+kUNqVEU5Vjc1JkTGqLctPZSZLMB/JjmwdZgOgbLs9t8leTSjBT4CpXDoyGKpPnsNXuumEK4jDhQULjcNbjzpTZuMxZVrAMReXQmUyc+kCuAH7+zZalkkRVev/U27pCxsFCDsyRWy6v0P+MxrBWGz9Vfgtk4POeBqk484bq4xQK07bN1Qzt9tCjwMQKMgKcKAclX85OIQzZQaqHiwg4tXIYGSRPDlFdO4daovUoEqDIh9wZ1BVm9vpNs3ldF/yGt5KZVPtBsld2WRZDdN2mrLmuYZRFb/LPt+/yFurH7mfFqWhQfuF/uGhKa0DI9HfiQAZfgomBAV4ichIBgToWU5Twz+iv2D/x6wfell4aIDXEYpliqIoJTgolJ0adA19gJmQr6QDppnA2+C6q8hNPMcu/yIsCpggqnq9KqJMbKADQ+wKbwUkpUMj0Th3mjHjn8SM1a7OibGUxncrlc3MGshx5BnANA40xc1FrbEbY9N2x77Jzk4GbKCeQCf6RSwPSU0WUJO18OI3eaX5QEk+pF1iiAAA+h6LEVw6XF0CP+3UwEVfDjCW5GPyWUB6Cn1R+0ImVv8bjAZ7sgI3F3sMBNSHe6XXBMSDgBDxgXsXr+CRYCjlmhj/0ELNf1wCOAp+ErGm4oISJMjUBjgFiTzvkN1QH9a7b467AZ+WCfFa8VGllt4xUXezx1goAtLDAomgBi2HN82+G4JvtFvyoAg9asIEIVcP/43g5W5Gf3PQH0R9ij5okGcApq2YDMFyfRCsS17pgsfIdpLyllwxJ14D78ghbXlyuFJC/HdvmyNvu1oAU+/zILcCuh2QNWSPd7gi4ll6SnhKkQy2doSEPlbPDj5nyeDrMux7PYiMnhrZRTDUQtKfYU5KxMT1F7x5b0S0tRkiB8KwOxJyCtlT9oRaLMfUGvq/UZENI0lQtJU0eLw8p8QuI7i2GYaXIygGDGRXgU4uGC/I4VKGRL2ibDR5H1rspHhM6yqxEIAmBpAR0Mgbl2lZN3n39zGE8RYRhfLHBYU5lHXD357wS3D/UJL5CVp+jZ3IqDhVfYtIMaQhYJSQen+RCnwmyrmrqkPUR0OSMfUF1uw1QIapywwceA2a0QBoeI+K//EgpFkwHmOA+uYiN6iZnCYT6bAcGMQR3jkjBH7VnRf0ADii4tMGLy7OEXe95C5WfefgDm6XOnK9c0zSPPfEgKfVt7MNIIDA6T360DjpJgk6ZXJOOgCrMyZNBeI6nt+CfsdonPkbJMLIidEuej4Afjt2ttHrjE2LPWENRxeGoMQWPbGk2EjhQoc2ABCcfT7smo5uXeAzKMT8BpYIUBq2KY38zJ11ico5E7aJQRyVwwfN2s1eVse/JL+crR7qgmWXBF1FLpXXiHU9Zz4fRDKD2dQmyil10A/Yr5g70blvlXd3Un3nbxD6tCweJz3KIZFAquoDM5+UQy4pd+HHUddoeVB0U5GwbXHQgV5QMpY9SC2BdKEeKYXc5IHh1WiEwehxvF9J1PuT8HG3zB9xaT+f4sSJ1tlJu+7TKBxXcLJ88iN6itnYmlw+C43nkHmJV/Veoi7v1IRP5lvsL43P5OYUhUxCa6uQliLnj6N3p4lQqMXvx4eXr10ymo/lmw494BLu+NQX8E8H+88Nvb0FT2hLKYNBdqlIoTgBZmFKvzKmecssNWFFZ5xWaCwfZc7xEiE8lqaLqdwCm5VPRr+M2Wv7vi4v/yS8+zy6+z1Zf0XVmlFI2ouEmyRS+lscYRhJMovU5YWSBAm1TKIJ4XcTbaHsng/fsm+I+u0OEy/n3z1fOGa50N5BdwbrEJtf4bXjiMcxCfuE1bbCwhx76rkmxtZXHkJLjkXc8gRj0PalQQsyAldIcq91aJRynir7gIdDZSo+tMT2iawWbOpqkRRa5KhjBn04vRlqm+ovDk1GVH6gEnSqs2w/KNJVoyupBpZXjhVhWqIMer84Yz1Q290C5ZqZbZijw9kFgdrbJN3tg2+BswamGJwPGnCsyhmcQuKfh+gcOOezB0chbOgRjKjGWyGBXgRLByGzgynTmHsqchxjcQlYisgk/SRXMlyTlzBiXBurBeAMnzXJrzxR0anXgXY4ndeZOaM7uzBbvo5G85XXUwmWSDHGBgPofrsOU9xLG7pg8t2/tWWjR9HgeBHawKbFbArM+MR0AlMs78Lped5Dy0oKZKLrPqys8m1e3ybUo8UKAFNmHSQFYpyK0mJ0pCmRSp+c+/8YDdYXnCfJwX/JpGObpXxXh5Qmar/Z4Fky0+NcxPgU6pEyxeNBaLJ+qkodGEsy1rQkN1E/hekj5yXqHU3YbVD3cYRxUbXErpkfI6sBjH6ZU6AjsZYmjqqz7mygJK2zX3oYHCKpqMEL+iZOz5PD05d/fvEFk681UNNPnUTI95BWoaAbh8RDPkuB6foMKxOLXhInMJWW/fXC7gEL/HfHoPOC7pCpM3KgWih+J1w5E53yyTUnE/96JZtABhg8d//SBoyTULVjwEOr8sRetT/wiVXtImZ8DvNnErTLMMku6MbSz5zcPRVTrwJe6Ocz6r8Q+CyTrRHU4Vdc37tmD4VlFAbkZ77ae1j7isMfx50Rr0JsT8ie45ycr16FL8S0l1+eK+V/5NGAZo1TZVHFq4oBaV9MlZKXa676sCn0Z5Ko1XlaTboNgIF94p0LquUa04O2U4zZ046aqEbBDDjK0iWPfjnnjKDwdGoTb/ofHwuDAqsqA/HHxzXSW/vAtW3PQT9WohJ1boObYFAkAJ67uAf9wY+Ntq/vgvC63XKDSux2BevFTc72tJ7pndt5h3mkIJ8/8vAsxvH9HC8HGEUqBpV4mtheFvqMxnsDjgVN4dJeKvpZvsEPzNKnEHDXLpTR4eeDty/bIKZvGhio6AtUWF7ghJEiPLCfdHgGHnXR3j6W51AGvey8wM9Dylzyuk88YOpm36r4RlN70NfVJFE9NX0Rhe+x0w47i5Ci1N32O0rTA28kt6bJLLiDXlbLvLr9/ZrsTHFgPNcgo6NMAWHPVrztk/HZDpQ3LiAQEnhjcmq66qSbUEoycywTZdwxeetv4ISF6n1+zbV9VFwCmLW+YhUZX2ZhOATsB1V+d/BKsY9ibLqS5Rz50HSV1gBayifeLgnJZdoR0lzYeuQVnOrgUCF2TBy5Y0TXf3T98d26naeZ7M+lIhFIHO5F6kSgFAGt32rPn3qEimo92SxhTMTFYLWmdDaN6M+o5LAlsxfgAleEMivU0ZLujOGYQnMmlFHz3YkyxxC5feiINnz/YowcJcUSwUzCP8xiDQLJJNwRm1CwB/+DdxUq1CCWPFIgLZNofsYklHjhx47/tYa3HOmXYGpJtnnJhJxM3x9GuWN5pYDLjBYHE+goD1clyHjhD2gYiCtOHw3cayD3Ujc4Fl8GjktM7/eA+eiwrVVo20Fxy08hRq/Kyt3YFH0DOVveT/1vidker7umOg66ZwThFvmu59h7nMgDdI3S2J0reGwwkI5kBWLIWXOLCb+seBAM9jZIX8F1Uuibj5iBUh916qlxkHLAy9XLHKpXyWSgFtI9dO1oEzN46S7wF3zUtvW6ipiyWCMEldqG/qYaME51LjyDceRllRL0/1vItb3HVwmNOOvSJp/f4J/bmdl49Rhz4Us1YGurp/+tOVFPVMb/FF1n8Zn4PSTRnJ1QkCpAEs8+qlYFNtR5OP5FnqnGnSygKbDYAIDhr5XX2O5YK673vbvu5Kre8qsiK2ZPZKQmObWExfmSnu+q1cL+MdEIs9Id0EKmVmoW1yCwbyiUgtPP+wePaIsBD60AX8s/wMTX0LfyX2nTL8Jh6pwlwuMZX5URruayPD/mmbbLtZSZbc+Ngs6p+tW+5tJmdvtiQLTWUzNGhZEq+fKXSYclsWQr54dptDnWaOzjmCZg+gNpimoj3lXPmonUbO52mnhEFTrrYNut8XUKcK/mgacHBMkzp/Im6hICIJNO6+MQxkqnTHJxAfHnAG+XLQSfFgDB/ke6ccDiVsovLZPrxNCJagbmCTYfwvMtl9gOtaKOzxm30e20yLvtmErOsk11W2G0+IuX+r4MiRUI08O5O03nvL3RaCiRiyv+opWV3yG9iD3nK8ptSLJyWE3p1FG/F1VukCmdqoaW+g5A6C4CueJvv+CIiS4H0A1sFsqK8oluCxWx0Xxxpk4qUPsaEMJGJi3cmq5XddLK+7vDaEuC+5xvCb26KX9C96QW9AGqqQf3KsXpZE0rJBvYSeDHUaWPVpvitPOaUB0lZqRHrvqkHe17NN3o3qr1V/c5+87ITV9RLqFlZ0JGb49RBgGBUhgJyO44jdd96mysLt2mY2WKme/gH650Z+6b5NHecGd3iBqlm8lqWkDl7KKC0w2MNSHMh50cfqwhSF45OvMhv1LsEWQcMytTuibbRZM0auqw4yQgAWR76gwYFu+lbMWII3r1kSqiHsu477s851dKtpAjiV5/8YSs9qgz1F3/SUJLqjGf4eNAY6gkYLZRsxns6WBEQuj5OGo+cXovqgG90+k/86WOJ4ysRo4f+ohPCpwOL4EiY1wN1UFs8MZqMSBgri2FucHQAIaBIZn1gbCjUHJIzcEhbWIPn2VP8nFGByds4NBlnwZ4O2Nmof45gWjfXsf5FgmnfbZJpKRq82MvxbEL2/Ar8OYUNpC/7prBOS71pFCsXi7uUjTwZGrxziWU+2Ass6w2kAxi1Cd/5jVsRHV3jWfNytkqwm0V/vYSvkbI/+eTZoEGCWnRggPpygrSa6feuT0bXbzoQeH1l0yHfuIGDj5EDTvs3JGCcbN+28o03VFEgZ0rJ8VQ9t2oaJ+OrKuZ0zSvnk7Cv2KWb5WgEwfObE7cn2whYcYEn0OWub3rnBSjHxdpMxBzppPYY507hdS+u3Jfo8LLaaBrsOixBN0XAsO5VkxTfDUGgLApl6s+wPjeaD13yYKqRo/QRer5vZSfXQJYBKzx9GMwdvoc+9uJD4FT8+S8QzjUf5RlPaEoyAIPv550F4pSO5yG52OQrhmfIUe8gniQmCGBEShiKwqMTxjAVejRIwomlLv5T643SQRwAKymEkju5i4uBHx+sdYKmXey95S9RiGyL3YHqBzrm1lEswilD6ng7azTqzXNrectP/DZwNSjNS9/6od+AcbcHRPNU/b5Lhu+B40s3CoJsZvE6TIfIH+F1xul5AWkTHhIDfRxfwtZAtczm7G6I5374mo/yjLqBUm71TNTVknwg1HqGbjvd8BH4rkPZaT8XyIRK+W4HNQAhFylgmOHYtgCdSZhIMQymwBswp5IpB7m3iH6KSWnZxfmE6seFfZvoBJok8MoMaTQmjg/oslN77nnRVxTp9KIvzqSyugWF1kJuVmGHSW9asyR+3GfsiAmMweHloHdb48eX22g1XkM4kEcvm3Q9mlmEd1m3dPfUcuxciBCchECdaHFEywt67pFKw1Hb17I9wYM/dh42swzk6DafSD1TSBWho5Z4tIxlyEWt/HRfPR70+L262VR94VbPTz98zcRtvdm3oLrqplb+CtYeX7evquYaL2n1b7wM4Bl9eCKoMpcNg/jrEAgi32x6iBDUq0x7vNA9HdKcxgbx2NT6T6fXg766obqHSu/RUY86PfGqGCnp+jg98LyOl5QWB4LRysuNXcEv54qW1SoZt9Otq2bzSd4Od/spONwqDlenT0/uYwwUeFfiZX2RdU1H1j/e2ZcnquCvQgr/paJzhKmn95JNaWYNBA+58NduzhV4X7Kvn89mxu8FSrgv2fPZue0peAZh6rBxyISnJ4gNCOVkS4RtjUDndXE5Q52mN+Tzq51l4uKL6ddb4TKS2KceDwjLLmczGLn0Rj7Rb6GZ/S2+KKI0SMlIsGlAfukQZxrg4Rj+yKhOO2GbXpz4obht9EGFQumD6b0F8/NwVv3cGuZfJXgblA==\", \"train.py\": \"eNq1O2tv20iS3/Ur+ggYS85KjIPs7i00wwGyiWc2izyMZPZwC8MgKLJlcc2Hht10rPH5v19V9YPdJCU7d5ggSKTu6uqq6up6toIg+KXLyobJHWfb8p4X7OX5+apr+6Zgby7/yd6XNzv5898+sE0meFU2nH0t5Y51XPR1tqk4y3c8v923ZSNFvFj8sisFq9uihxkY4o0s2yarqgPL20bCRoI1Laszua9aWZUbVtb7tpOCtR3DIVk2NwBa8HgRBMFise3amqXptpd9x9NUg7OsaVqZIW6xWJix7mafdYKb7/8WbWM+V+3NDWA2X1thPu2rTG7brjbfxa6XZWW/HSygLGuLue/LQlFWZJLjjKHLfFez+0zuBh7ZJXxVE/KwRz71+OvmsGQfsj2OWW6avt4fWAbi2ltas6aAAfi7L+yY8Oi9rXjWNVpsw8mYnd7YkQ9Zk93wbsky2dZlnqKw0gL2XLI8a9qmzLMq3WVCEwwnyiuDJVww+PNO8o5O4DPP264AXDRM2gSMXGa94J/5rz0XkhdqbtOXVZGCjECVpFBjdZZ3bbp9mdZcdmWuBu+yqkRRplJjS0F7tuXNchEtFov3n37++eIzS8ypxjdcvoePvAvTtMlq0JNocfn6n18u3qYX//3ul/TNp7cXAP6ff4bFBd/CuqxQqDXeEE9qzYTs2P/QMUVs9SMrylxewdgST+h6TZQp+BThASOC0tqIJulmOBBxu+dNyBvQZyAzCXq5Xf01iPAEd3CWFVc48U+59RaKfguXMc5BUNu2KsKIJQkLYjylYFg0EAS04FyMnIUKd2TBeCW4v0h2B3+ASFCne8jqypvj9znfS/aOpi+6Dm4qMACjUxQgUsHZ577BO0CgYfCv1x/eayp7pTBgPH7tS7Ag7PKAs9+zf3z59JE1nBdkHfg9nA0rOAivAOEdQGKkhLDlPOtIciyyLU8n/INcwVKwUoDlkVmT81BrEp1uNLCgSP+vrOoN4UaTx8S3gK/uhWQbDnaItZt/81wGajvNWQFEPVjUwb5rEYZ0M1iygG6T/cbv97wDeTUy7dqKhgSIAv8v+F2Zw8iACsxCumlbgbBgoml91lWHVMiWDAiOlPUmq5DZlEShR7c8IyMKZhyIAVZcvHBZUzDT+lZ07VcxEApWNavpu766+HEwLopgIQgjfHyF/+qrqbd4VFe9BKAGD0yAJvEiNNKKi3K75R0fjieyp6cXnTiobfBBI5bzJ3bLD2LNHjSmR31Ux2yMoUCfJ4is0fi09dC8pgXPULaG5LUx4YPJICuyBZ2UYFY+to2+hXV2X9Zwjru27wRIgyA0lisryuurwAMMrhVJeNDpBswCHGVdNr3kJ3HMgBtMKF2PlB8Sdu4IWvGOdNNYL9Dfg/rALgXu6S/+jr36y/l5fM5WsyR+x/4Ck2bfEa7RxpOrqPmJ5xCbq1gBDEQxcCMNtC8+xbM5NIcNn5aFMnng3CBssWcMYK2IeXNXdoAY/E0YXL67vHj/7uNF+uXiy5d3nz6mby9ev6WBi8tPb/4eWBlPkA2czlADbIXj4SWKOgTxLfU5T1CCplGEEuM/YRR52ksTdQt2EP06+JI/TjbWqm3vBIV3MAsxFg+12sHHGR1fsq5v0rIg77lkylZQ5KBHrNkBi1Fn7oyEiI3LNcaJC7oq8EF7WTQuwjeiaptgzUIiRR2CHowMFZFj0hxSRqvcmcij2V0/Q/gIzxxENMuxi1exnZYmfkKkwLmLeAKyZKuXEaBGODUZRa5p3WZl1XdkCR7A3IG1C9qN4N0dR4mZj9rZ5JJGzcdHBvcJjeSShQOkmY1gT30eMdBTC9AfUGoDyP4jGRAZjTfUnDTab5zoVMAmeVtD6FNiSkFxlMRMAk4VeDH4jOnO+w6chQRmrdyuAj3oCO3aCwHO0cqYlT8kriifTadBzR40okekve2lKAvOzuP4QWF8DHz3oYD1JetAKZTDIRcuwgnldCXM/RgN+xfF4mIuQ8MStAs4PtnCimbA8MNpG+zIIas6sB8HigwxZsPc0bhcUAlFxCAtXxh2Qy2O1PjTEsNcYA5sw9pdsDXmP30w6VXctF/DCNxBtyWD94ezf53VZ8Xq7O9nH86+/CF6TB8wR4vxnz8B4I7fX63/ev0YmD21JaegCxKPDAObsMJU92ZTp3e8EyRvMlMYcskdciyGE5jLDWpet90BTkIlZTFsIXsw0mo89KTgRocHuQPm9KZwMU1OGvszoWfc1BRemrLi3cwqM+UtG7MI68ZDDrROJx1gPRKnZixN/eB0f3Cgm/0RQJXJuiwXxyA1Vx6D+oPHWb7vPZiuzUFrQHEjVVrQ45Bu7jBm84TS3lC6CyhAapC8ICJ1hHYo1EDJL13PvYPYHcTzV/+UQS7mkW2vTeroWUCKFjojngNpJewHPivdHDCcU8BKy2KatL5Bq/vXrkSn3pt8V0fGyrPjcFF2a8pml06WO+ftaVo5zCemHU+vXNSxOIBmq2wDmUatEI4wA3fXPg5MnQSqrqBLqGdVtnIaZv6Oq/BD9vuKX5EUSBZ+yg8igquthcVemKMLRjBxfQv/hiADsC2C1AWdKVCRtrdKe6zUbQKL1sTPO+xs3O/R6oVzkZAmJkZWHf0YJ5RrFpjiHSKVMHgqpVxjBdBF5+eXa0ZK7ADMpJuwZQN5QzAXTtn0cwZoJg1dM5LhFNG2ywyel7FL8Caj3PM5APxXmHSnyLyRbwah41x8fv7yaFDpfHsycJwZnVljLEjFm9DT5LnIlCYc1PTdgZu5EAA9M3oEt7mPsIg09Ldy7xO1nEMWRSPlyqtMCNoaufJuugvq+GQAfZ6LVhGxOoarwLWg1zZAHm6Urto5F/qFuk36qlF5zTVhswvcooizZFxIDQfES7t7NA9K0lUYh3xkgDeRk0biQRgTn7fgevcdR8NTDAIzI9q6+8aerB4ZyN/d3AF5JVIx7AbYr66VVQcHjZqDmQYkZ0Cxct5Y2iX56soTsNduIaSxgyKrwWKDLjXllgtpxz0VU6NOjC/avssxozeyAe5we7cWiwmDgouJGch6/Gqnio9/Amo+tvIntJsmX/hsSoAG/QppZ1knyy3YI0wYTE2LPag9TLqgSgJClo3KMkaa5xGpGhUxHjvKJFSYlu76AamSfgwC4U0RTkCA0LaTqRUMlbXTFPGmaRRD5tVWdxAzxeqYHW2gC0SrnZug0Z3gRK+Z3rsxVx5lyxnMrnYZ/o5B2WIeApt7gxq+LbGWopbBuYS6SkcdkvVM08QaFFP2OBZHkVtFHNjioRGI16Qxl2I8TnorKG6G+LjrsoOOTABPpfTPq9yr1kx2yzXtpoTP7yBvwzKOYowM1mJoB8zMhr5yK06W3qDmxR/02JmZUhz5ExAHV5ssv02qrN4UGVMmLQeKbiCAXRvBx+LQ5Km5NqES93IE7fiPaFy4ROOzcBoYF/QfxnwWUnWS4q/K5/tCCH5C8RhpwbIXVkGowMGL75kjQRU8pRA6xfIeG5SF2xnVWS96K2AmWI4bLGnZbNvED3QmDFHcpfWWAqQQEm1QF9P4jD+i+91nOR9V00yYOe17IYLYDTzB9tGY9SRkePHWloKs4lDGHlBf2cbA9VUwWRmgqZ9H6m1ZZ/e6IpJi7cfUBeY3fmqRX1Ger2isVrBatbrFClevzGpTU963opTlHQ+iCcd+mX6OhoHvIwAqIdE9TIjQy/yNOhW7WcXveJXcgDOXsgs16BJNji31m0YL0EHQEPhAyrDHvN+5HJj7ZjIJzsJM5FgyiQQ7C2kBehX6pj6s4RNokYArGAmtqNHU9Ji+uo7HsCNY3WwWz+wXzvUJ32GDrqosyh+TP0HQ/sOfmar32+aO2w0khG0v9z0mL600voukrsdBzSLXoAPMxKIPEneQJc7nQZJD7y5xj8Fp6V07ch9MgL5xiVng9M+uB3jxagwnXo3meYPtiiJtIQTuyoKT1bD3AVJYvOSO6Ullm4pX1ASmO+QeqTKq1MpRNle7+lRNKDnqyrrr49wVaiR0oWxgpY/jeVZCrdUmGe1hGFwSFFpcfg92szowYGB4kvJW4aMAkmf5TunIC93FQJONToiJfVWaBu2mx94wEOa/RbBx8pK51lBXMlVt1XAxzpevPfWKi/ZrQ2fg9VBc8eAsd6VO4FO4cmtAwQSO7O64ngvYzu0s2BhJrpSibb9U6cTdtEdTSuXAzPMH3AhgbNsP/zyVXdi8YrCTp2tO40iDmURJnU+sEpzRV5Uzm7GT+fRw1LGXDExW6zRWj86lxmjX5ouTkWteKTDBDEb3tsNp6hTFN1W7CYPvdEoySimeE/dYVIvpww5SoKU5fO3uIdA9pTMnGn4jtIaObzwRfYVmZDZoHj4IkJ0nsMF8UYpRZULDYnDleGPPWnxGHtBYACZ2JpSbwDJ4xSVeIsv3Gb6bMAzNtEYmvZGjnRrDH73PcQIOHSzjIYD2/E19C7V6QeyYfCPDkRv4oDHSG8QTisDtY09O91ZnIqDXQvAOIbXf/cK7ErTgN5AQhbNMU8uKlqvIS7/cszYYYtxBmMI5DavB2Z2fXVjl0TmEkdp0JZdkMlDpemHXBdRvSuGa6UwnmKKw0Xlqjhykfzy9G+086IM1SZZYX73FVYBeBbyyZ2Hs8IyqH+FuYgmJVUN9oDRwwhQ58uMCmdnevyTqbnzFh4S6jWdRZ/LYJZkKmzKSc3VVhg4eitzt5y2edbh906g3SkfalE4UpZ8h+D4Nu96PC/2+Y+uC79r2NvSycLvHSSe5ZIpIcorDM57hPjUtOHnIZCe0zYV+2oS6I84jGufODBBzypFoifnm2Uo7GT7OZu7JbAavxZCMxOIDOdY+mXEAHuyMF0ie67LH7etEDSwnDqkXifpvzhuPWLlavbx2420jfMrPpsOTW6saMAZbmm2BOvekrCOZHrhPyLPNnWbtSUM/IJw86CTfd1f+ZhK1G97gOp6WTQ5uDTBZkygW3spTkOH0xedcqeh3rPL4J328svNUdYcqPO8GDofH56q2w+iYPXP4vfugWoUGQgVQwZT9p45uuuJIFWi2EjSjtbrCqZ5ig2JPnmeH4/QgmVz1J2zJk7fTa54l6qmaZy6uRv21I6ky/nCgA8kmbtrlJsxXgQHRkdkxTGj/k9H35fT5nX0clxx53OlgP1LMSSaFIfWU61hxKJqKHa5954p3Gm26RNDTRrOUnjgmz35I6uw9fVGZfONbUregYG49erYro40qJMKedFuMcmm3emVaewrQeb2lV/7oBth2I1P0xzgbsKVDvTZU6xL1XzRTxPKjdFVSnfF7vssbeSyqpoIdN0BqQNcVfNhR4SD5hvTCR0RGBzcVydXstiYyHSohZup6DhMlugkEsIgEG2jDuuB67N3xXo5+tBFiX3cu3R7buSHnS4aPPsgt5/vhSbYJXaY20SpAYj+NYwHtHeZ/kTKpSXpR8pn4njX8Xpq7yb6WVaV+9DQXJgOiJTW3jadnf2QvJ1Z7/JMUZbNVduIm5U/5DvvMcrTSpn344smjZiYhnE0G/aMefgGhGACh4cvMtpdHUsG1DTUfRrQ92rpE8uBS9hiMzkyt0333/1s++f/MJT09eGOrB+q3cfZHcJ4OgBTWkFQBooF8t7P6TUnp75OQnhTN4ndJQr2m67nuW1GjKsW6snrsOe1drc0DDEGW2QK87m56jNUuaQab2HlXUtSXpGnR5vijr2FlnBUFbkNLsNWjC2fYIN9mfSUTXUp7QbZGV+ROIfDa+SusWA+4sFh6crFqJYxWBWpUvACxi9ObA8QK48zn73iktYVHdNjzhB4YPxuZ6iqsnCrVSrYr+qWPemKVYGzQ4W9peu4/99UY3XPXqoDOTynBUNtWVunLAfS8vrgvZaicsrs6wvWgleanfvTLuDRFbGkamF/YIOrF/wKI5jiY\", \"viz.py\": \"eNrlXHtz2ziS/9+fAsetrSEzNCM5ceK4VqnyOM5u6vIqx7s1eyodCxYhiWuK5JCUY43X3/26Gw+CD9my45mdufFMKSQINBqN7l93AyAdx/mb4FEiypJVC1HG5W4B92t2GZcrnsQ/8yrO0pLNsgKfs/fxfFH99YcP7JyXIolTETiOs7MzK7IlC8PZqloVIgxZvMyzomI8TbNKUtjZUWVLXuVJViXx+c5OfR2sSuE6R/O545ma86m++leZpVb7hb4uF6sqTmTvOZQDId31Z6xGD6p1HqdzXX6Urn12zJOEnyfCZx94jk999kX8tBLpVPTwGeRrvGK8ZHlS6efpapmvsSzNdVHO0wgKsF4k+7aITLMkK0rNxvts/jErlrJWeZEIXqTBUlRFPDV1+Grqs7wQ07gECYZwAWyH01VxCYwX2VRe7uzsHH96/+k0/Hz0/uTs7ISNmLvD4M/502Dwcu+HPceHyzf7+yeDAV0OBq9OXj6jy+Pjl6+OXtLlyYtXb7GCarr/4ofnJ69Uffyjy5dHzzSV/TfPjl79oCocwH/Q1Nt5/+7jSfjl7J/vT74gH84uVtiVvwH+HsL8fjg6/e+TU1khw8ISf/4Xf97gzyX+fMafH/HnL/jzGn+eQOOjH999Cd9++ngWfnn3PzjY4WDn7N3Z+5Nm4RB6+TE8/vvpP07Cz5/efTzD7vZhHDtHRRXP+LRCHTjn0wso1+owHqPW+KysionPPmapmIB0IzFjIRpFiGroop4dknp5bPc16tMhyexrXC1ICYMsF6kLupRFoFkjZ1XNdg8cD9ViAQqSCFkf/woB9pKSegdJxiNXVvB0r9MsrcRV5RarNIziwuq2WuXAbxRPqzFw6yMbwHISl1WrUJfKMVEx/cRpNZGMIHHoaBbPQRTWQFWn7Clz5GMHL+vaAdaCGUEaC+giK9YbCSjNJgqqrt18Ka1wm/5hokQSqgY2jWnCyzJM+VKUQGeMF4RaeOEzgKSUlWBVInJ147gSy9L1fHYh1qOEL88jzrDsEIXj4tV4OPG8CZGfxSlPQigsCM+giyW/cl2sCYaZFdHYMQ+diUddywfYsxoz9AXzyldJNRpIrpUGuLVKGPn6pky1rgussdaFMK1u3XrsLLMIJIWVgJ9N1YK5qFya1DgCE1MyD7CVZzVqjV4+MGpaxVUiXPQIh1K5qGt1LWmrmxYdkjQo5Go2i6+oCsjVcUjD4UbqZ15k8wL9Ez5i8aw7FQgCAyaSEmbcYf9mgIuFSKu6yui61ebGIdJgWwUHutTqWrJxQ33Ia0nTceypmjnXONIbbEHjpCs5Srxs9TQC1q71EG6uqUfoXUmurNaJCPlVXLr4c4gOJji6gmll8wKldp5lCTB4VqwESQVBSYoF6wdVPL0Ic17wpSQwcs6zagETSWZSxj+LURMwpdahclIN1E0iBGoQXiE12dD1apCikqCECoVy5u7zfa/n8YKDJmF8oCwSxEiDMFVlT1Dk4ngQmFJBEhhJH8GTfMFHg+C50SysIYUElhaJK6UwSw6oq2XzlsMkkXCayHdoz9q1YcIhR+wcsobXHBN59meWAHY3nniT2gwcwzG0t3xdo7VV7k1QCMit1CS7iRwQe/q0p0uvj5TFxpIXFwLHoFxpo39V1u67XXWvQ/BSFGugOQiGe52+UJXg2bNgryWNr3EECnfIhsGzffnoRk8exH7xbE0OEzRbB1jkY8EVTXkl5oBqChmmyh0fso6D/jdpfUv9YWymSQ1ToNfYHaq17NY8IthUDYilmgOjbSW/FCGAIsSwrgL9uTTJt1ToG39pnLEsQriU46DbCoMJaBgFb3jF3xbogXZsBjYOEtQZ/1G1y8tQab2t6hBnoSjIq5MsDzWvwCKipOU4VaEEMB1eNmpoz2zTCJYX8BSkhChajqSxCjDeKswu6NazCW5bPcc4O49mPo2M5mlk2H6KwIpyvAmgnuP3Pohm8EAPw3oA9MwAABMBgsKEr7NV5XqmGCcX/nWJiyiPR88GA5+dn2dXIOQppD4jp7LAq9EEee6pSbzwCOZ4dO0cQ+SC4AhTjpZC08icD1lkFdx4tX4EVRYC366WBYZkMNcjM+sgAogJqxCUGtKIkfPn4MVMMYcqOU0ySJmAPVk0n2KGkYipHrM2P1eLHYIPow+1vbUqG26wttaNdm2FqePuhJp4Ocq+piVfQpDqPuFFwdeAAGkeQH6EN1YIWxf6LAgCpcyAY3NSDwQ02X48mBi3oh7/ZcTaUX4ntqZeXOiFl0THveQJ+lKECrokF0g9KPIpuBGyI2i0SmNALWwOWFfmfCpcUBrV/S4b+h0GQLcg4RQjaAKu6sVzryGyDdyMVaeTfrakTDGRDClRhOBV5n5lMzPwtwaYXgQJfRNp4jUFNzp+83sC4HZ6Yik3PLXRz9Xxr5LFT6u4EBFUstxyHT1jRgjBEua8FJQ6EDLFabiEsDkOk2wOik8ZI8ipXViTs9uIogCP32hhilQ9Pi2ycDY0lcy9cmkEdzEwRVmKSiT0QIIons1Egd7NlbYNprhapqVnNFa1tdSTx+CU/4FTfYK8uLNGYgRh1fSiZHq6d2m6YQ5EEoElXStyNzr3gbxxLvMe0wFmJpKZOMmm0u1PmklKw2/KuAC0rkBa7pDUXFLw6powlB6i1nRN2H+NOlXQVFrViKJOrQBG+BXxj8hWrs5R10vk4Rk5Agpk3eFLn+0He4qbnKcQ15jVDvxz76EpAM1YsHtLgQws/Q3kb1OqoynMF5+uW9fkCnvobVQ+4Amvd9+2r23W6pAe42ufuZIqqKCPUCIvSFo+W1O0jq4Ggr3Kw+n+Oc5dThmHlKgV+UtCBEco6GEwgImkuR2bPijKlORknNl+bqghL720FJMbKOmnzSwCFUQqZ0Ol/QbPKg0aOWdYCGJ78sTOKAaWXt9OtGbdkATLjSONV026Q6/HsnxWQydIXaSrJd4KV9mu1wxViR9+dYlk3TrvZpS8jJw/vaA/p5tDmZh8BCI23J6KEvpzpITRICFnVhkzuYOmHDCZkzm9UpsZAD3ZYGulrafdFfXoOj9kWVnRuquF64ZOX1LaICMV1dX6ul3DRMxFGrmm8qv6eTvTtoK8VS7HqobsvFeYy47JxTp3eULvLvmQ49ZuUfuHfO22n44VHnCNGBNjKN1KNuJMOoRQYe8g04YtRQSzJvTtdi5EAK2cPOBQKwYBAdV026Gi8mkY6fbnHfaKoAnjGwGyRULHyDIP2hwfm3DWbrtNRPuExu/bbE8a4VcRlpABRCtQlN9Z6DUeN0MsM4uIQs6k4Yrjjis2Tvgg2Le98B1gLMtaXRlUMpZG5Zvg+RGgpQ0rnZ63a92PE7unQIN9UWrx7XDRD1VKQ2+1y1o5HSV7W88tNTbMwFiWj6vJ1vr1r67SzayhHmQpgHZEMR7IZ3qRZxAcm1Kl+qoxaj6OycXFxAC35y7EutTabUewgE9U0VN6qjb4IKlRte6xxGjHCap1M0rQ3Hk3W9rpcIhm+sK2oXNe3GalXXlNdLwxlmMbUwzUSlJ7JAPtRDQXKlY5x1SmEZkMgj2DAT3GrHMaM1NF9rXUrnOs+uubSPaaDSaWyU55BUTdFqXW6DtP+yjjkjMug45oh1IHYWZntBzt7ZsRHZv27BwyoAs1HrDbgs9Rl8ltuZulHiwFT0Gt1EqHysFktDYIBgoOlhAFyGwUN6Ig1MM9kN0O2kjd5jOh6RsG7L0iNXsOJGlALV6uluEiWxUl7mQ9Yc9eSNK3NQOLzMNzAaohwmWcriqhGr8YNMwrTIWIKPfHjfxgKuLErQfzxIjpaYNpkoV+xNOoOaLXOpYd/GIO44uBkIe4incG55B3S0HOAH7v9BhQ4ECMH4PAQHC1sLRIR9ct4d44dzoZO16mlU0Itng6XYBeuxArgn0OgQao78hZ5bkoQBFnlT36g290Vk0XdIe/kvudEaBtEZ+vaAvqkaMv+N/eVH2ou1ryNJ7BXG2zky2XR0PdxN7LVoBXL+ZcO2WexJVzyOhfdGTILdzL7W2gu0rxMa76aIpj1QisU46NKoFZjql8Qnu/sj3EWjeNnJVqILyrRQq1NlGnvA4YeOV4Zosd61oStJZ3el04jlCO9UqutHK57IR4Z5FR7ovcBtQDx7G/pf9DUBzKtdoGQYCYQfByD5Rb+0bb9epR1473jvFbabtZ25DZCy5+ad8opwGTbjN9NF8YQ4R0kMhtpbLoq6/Y98w162fIOcnBZ3opQt1Kv0OUAwk4nvZRPQHIFs55vw2C4EkRxpJs7rRjctwzLt2rhvlsiNuPSWXvjbxkJiVzoXdGjHgPRGHqn72xUOTbg/Xb1h0ejI1duNuMj6E6Y4dAAtWvOucIZLG9++KzFFPmBFhVJwxae6uknNJDQ9OgXPBcjFVkhfuu9Pg1G+63d1ukpwe60Phlo6py0K8kIiwKUS6yJDKxCDLHU7RZ2ScEP0/ZntzAljwg0UYERAdssq/1ijX15TU3g+U6/OY6xm7NaMdA01ftJo16iPZ0VuSaWhwGz2fyrEgtTckhnrBBHJZ7O153XY/8huwCD/JBf1gC+RIfOVMBbgUXkS/tG6Na+GMC0K8LUFRiQY7htSVZeXJFmnfTl4IHWlGoTgOe9mwi6RpNpXkEF/ttzpX0SbEmlRL3GshlSNiRaK8cxy1bLc6xJqMmnUlqchunZHQoc+NBs4J/bXuzmi1e4u6fa3b/9IqVGrnaFxpZMEnnBqG+KIoS7RgSz8s4EiMnnoNiYWQUp+R5TIk1tFr16qX1PmbIyoAdsCpLhKulPDQ0xCNwIo/iZWlt1/fRlnYaSq/ldmrgDKfgQcDv5lkZpzN5nYq5um5Tbguyh+Jd8lNwg9hBa1O1CpCr3z/wtowYpGkROWx4cKB375YybSPTBREtsq99fE5BVSCdxRu0XsiBRgP8l1+hfHmZi2k1cviqyqwTBmTI6OupE2RwVHuKXmj3WU/f9aS1HXMdVSmzUEbYdrb3qq0d+udC4CoJTOP0Qa4d2b5n06ZHP82+7n6slak26w8ksMdfi9M7b/LIhUgpRozuWC9vIe5VWM+gY09nDbF+ffan3yBzSvx7DvroHMPujw7UdChEs/tQiGaSgjxxT/sXs7heBL+LgDVkdbynORjv0YjTEaHmOFt7As2eO5VvOSVjJjwA34QR3120wPF+G/YseXlB+gXAu+SBvA3xbY1Q/LTiSe15MIfbCFeyncaoy7iIIbhUchqpVxFcgqyhgixKn+iUuOXcsNDzEBUeEc6sEVi29QfDMQgp2uDFXJgY9kUnPL8KkGm9vheaQTyEJ8f511vwy0ROIVgHLzdvBXaQx+wJQmO9I2jo6Oigxkhj5XWdrfb9jAj8msvGzp9536YTLq/DCrpvxsrrMC+y899Y/IxrLoe974jgWpNMdvB0toQb+8AbIMHeYDDUmITHUyB4lPV+FkVWhkl8IVxsbNfReeRg2z2LAQZwLw32pWt1AlDFc6/ZC5P6yRxZrdvoBah62Ua2qANlOUnhOYiKXlOxz+DJZx4uz1hUvWYsf9A4A9UgFwBwutS8VYxw2cw1Yd6alRpPO2c4zcFlmM44XQlrOKhf4VStnZmBYOn40LfHMbFOJFIa8GzPOp6SgwpU9KMzRzJOrexug1nf9FtTKKeQkqC7WU1dTa1+GqExUTFdwTPUTutwaLdJvaPcbazXuvTBX+ZeU/+Ui3udHeSGjqA6Wck4GoOGu+v2aip0CjdyaeIK8aOyCtZUoMVliivvhhQTugJBVfpoVZ+ELTZqY/qepjLGhD8nS/JZVzi2XUGD4c5Gxarnyu+dZb9nbnY2KmLbfDBfxfgjyFa4w9Y1Jr+1pqv8cMeoggI8TWJ62azVPVrcaPttmryMaRo26vI99Xg7Hf6Ane7qXS332mLCKHRnq3Hzea+9GuW7iu0s7c6cX03HH08vOzpJi4PGHA5bNiKnqDavp3bltvlZ8y4tz9Dow6VWFWtCeWtC+eYJ7T3AF/RN6G2Tyr9pUuVqTj2lej7bMmgr9RhDAvAr+l89puf01xzTYetMYr3zf8rTKFuyv64gm3E2ROYUpLLPWRlX8aVgp/c5rdOIzh9Gohmlf0rF7mW5e4r7e6efjrc9H4jbt/T+uISKcnSNgKhw8tD/PWzW1sEvHvbr7ONt2LXNiz9CxPz/KayF5KkVwwKzv/UY1nwhwWfyEwmtEKD/Cwp3R7bkfTRFujGUWs5fVzLPN0az/eQ0JMroc/sYNudRJKKwMVoUJWAAIIXtUyXSbxn9SiabfsTw+sAQoSMiv8u9NbJbw4ZbJ7yHrn/bVP6eY95fQvO/WevvpfHNGPhB8e7DjGDbKPk/bAu/sh10bKATkJ1Kedw3BvtsRPbtoZehtSuZ+QPFYSaiulcYJgp1gE6tgrbDMVOh9cr6Y8RfDz/5nRWRwBTOcBfgW5dqy9MdOyWIGAoccxZugtsSU5gb/OrOWL6Fjr8TAE2BKkVOFFApy60F43Xn+Bl1rOxvIfBFb3ny7Pk9Tl7j0t8LuRctydG2zvMDzzpzls1mwJXe01aIqBCQTt65u7J7zOvoBUB69W8A84xrshYM4eeSrBtUQtc0LbTF1ra7Z5+OOOd0aF2fOlu4a/a94Y2YH6vTMD7TRCV42wy3TppR2fYnzbRIoH+/PlmOWInctdjoe1OO1AeXYSXwqne8v8fDQoMDH99+QqEAMfrQCYS08EzfyUG5dOIIMeLapiFP+Ww6jXPQv/0EKD0GOmCU4+9IN7+b3EBoPrqmrwVhqVLe7yYIPJ5Dgw99fZ6JhoufJyrQrl1v0gHjeEnL8sFgfwNOf5nKsyP3g2l5Qs5V3D30nN1nUexKSm+HIFSjmnBtVPC3eOyuBkoyN5qFDZg6E5y+dCe/1Mbxxe9N3wbZ+DGQ1kdDSN9CqeNWsZxQq+B+3xChd/tsqu1Kvwh2V1luzsHaqG2P0UZruRcZLASP3GcDb6s2NYRvm4FboAwMSkiGkFpBMkmKju9m+dgWG72XbBfQIf5GAb4GA0M279M23rlZYG9jR6kMvXAD9/awoOgK6I0kC/2IOtgGTVFbcxryXj8uXG3zPm0HFt4q1h8EB6jm4IDzBPcSnRAdEXO8+qDyf+htOZnrbjhUq3eD8XsvhVgC7gNPMAZQSyt6ktb0GO/MZRnGGFjSWn9S29PG4T3p/wgItLcG4ber2y+t3l61/WJgX+1JW0hyytShmRYOWjKyk2rsrLHWZ+fC/Q/7T6XKFHBDAKvOw2iIlm8oQyX10U+5nGe3mNwXYu83nfaBsV9vTvsO/MoW9cmbjY3tYw/UyJpA356wW2hY68APJtHNYYiUKb9t/D0vDz2GFdDudy7og1rWd2y6oUE4xxdIDpnrzNWbJGcZAApTdwQL/q0E9Ms/rnyVRG5iIaEv9MKKvm1TAuksV/KTfRY1ooOvGIaRAHTjpbA/LsLswrKK6ko2ZTwv3RME1aT5eRliJSINBQwKwNfhi9J/O/pMi8HWuw0W4zcmP5IILf2/z7TnIh9JHy4h2euPeNY5Qecw06aATU5rBx7oy6ET7SBavdcHnCQfDZg2Pe/8Hz1vMbc=\"}")
expected_hashes = json.loads("{\"checkpoint.py\": \"dd2a1c885bf89a52c761e6dbebbb81d4c6e4c7b928c1ba89f5ef7b0c2f8b84e5\", \"config/data.json\": \"ee4cfa4cfa4220ee7a11b7de25cb21bd60cc332553bdf817557b57ba16503a12\", \"config/data.smoke.json\": \"ae419dfa33d6f68578c0a381e462714256c92d9574f200df1e3bfa8e985fe84a\", \"config/orchestration.json\": \"f6da6408ca2fc51955320546b49d54386f87e80953060877778dacbbed0474de\", \"config/report.json\": \"f20093bd5f0b70c679c3836c67d7c441ab5c1451281b2e920dff2189f20b06cf\", \"config/train.json\": \"0edd8845f2c4e63cbb118ee50f41dbe141550c5767a3ce33e241d064e03c32bf\", \"config/train.smoke.json\": \"9ea1b49a57321c634a6464acc9eb7a62d81bab99a135b3e6d67ed3e635f356a0\", \"data.py\": \"8027bf33d07d1da0b380032b5031aefe3ee4d92cf3e327d7e052fcab4e2bbe76\", \"make_report.py\": \"3c64f197c430cef69f168f72c8eb6f6c29946f04fcbe3f96efdc225da5156f6b\", \"model.py\": \"f1fe5ed2200bb506c3d8d1b2d02cf67899ac6a1ae4800f83c78b81c6e136fee7\", \"train.py\": \"ee0fb30db0404dcfa4056750fb5f52761ec553b58c3789e8ad6d74c66ba79dd1\", \"viz.py\": \"ec942e97f922686dcb62e937ac410311bf122befaf1e8bc500f941d809ea2c07\"}")
for relative, encoded_content in encoded_files.items():
    destination = SOURCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = zlib.decompress(base64.b64decode(encoded_content))
    payload.decode("utf-8")
    observed_hash = hashlib.sha256(payload).hexdigest()
    if observed_hash != expected_hashes[relative]:
        raise RuntimeError(f"Embedded source integrity failure: {relative}")
    destination.write_bytes(payload)
print(f"Extracted {len(encoded_files)} versioned source/config files")


In [ ]:
PRESIGNED_CONFIG_ZLIB_B64 = ''
if PRESIGNED_CONFIG_ZLIB_B64:
    presigned_path = PROJECT_DIR / "s3_presigned_config.json"
    presigned_path.write_bytes(zlib.decompress(base64.b64decode(PRESIGNED_CONFIG_ZLIB_B64)))
    presigned = json.loads(presigned_path.read_text(encoding="utf-8"))
    os.environ["S3_PRESIGNED_CONFIG_PATH"] = str(presigned_path)
    os.environ["S3_BUCKET"] = presigned["bucket"]
    os.environ["S3_PREFIX"] = presigned["s3_prefix"]
    os.environ["RUN_ID"] = presigned["run_id"]
    os.environ["AWS_REGION"] = presigned["aws_region"]
    os.environ["AWS_DEFAULT_REGION"] = presigned["aws_region"]
    print("Loaded short-lived object-scoped S3 operations; no AWS key is embedded.")
else:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    aliases = {
        "AWS_ACCESS_KEY_ID": ("AWS_ACCESS_KEY_ID",),
        "AWS_SECRET_ACCESS_KEY": ("AWS_SECRET_ACCESS_KEY",),
        "AWS_REGION": ("AWS_REGION", "AWS_DEFAULT_REGION"),
        "AWS_DEFAULT_REGION": ("AWS_DEFAULT_REGION", "AWS_REGION"),
        "S3_BUCKET": ("S3_BUCKET",),
        "S3_PREFIX": ("S3_PREFIX",),
    }
    missing = []
    for environment_name, candidates in aliases.items():
        value = None
        for candidate in candidates:
            try:
                value = client.get_secret(candidate)
            except Exception:
                value = None
            if value:
                break
        if value:
            os.environ[environment_name] = value
        else:
            missing.append("/".join(candidates))
    if missing:
        raise RuntimeError("Missing S3 configuration: " + ", ".join(sorted(set(missing))))
os.environ["PYTHONHASHSEED"] = "2026"
print("S3 environment configured; credential values were not printed.")


In [ ]:
required = {
    "lightgbm": "lightgbm>=4.0,<5",
    "boto3": "boto3>=1.34,<2",
    "requests": "requests>=2.31,<3",
}
missing = []
for module, requirement in required.items():
    try:
        imported = __import__(module)
        if module == "lightgbm" and not str(imported.__version__).startswith("4."):
            missing.append(requirement)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import lightgbm as lgb
import psutil
host_ram_gib = psutil.virtual_memory().total / (1024 ** 3)
print(f"LightGBM={lgb.__version__}; LightGBM device=CPU; Kaggle accelerator=none; RAM={host_ram_gib:.1f} GiB")


In [ ]:
preferred = Path("/kaggle/input/cicddos2019-parquet-per-classes")
if preferred.exists():
    data_dir = preferred
else:
    parquet_files = sorted(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No Parquet files found in attached Kaggle inputs")
    data_dir = Path(os.path.commonpath([str(path.parent) for path in parquet_files]))
print(f"Preparing deterministic leakage-safe splits from {data_dir}")
data_command = [
    sys.executable, str(SOURCE_DIR / "data.py"),
    "--config", str(SOURCE_DIR / "config/data.json"),
    "--data-dir", str(data_dir),
    "--output-dir", str(PREPARED_DIR),
]
if os.environ.get("RUN_ID"):
    data_command.extend([
        "--s3-config", str(SOURCE_DIR / "config/train.json"),
        "--run-id", os.environ["RUN_ID"],
        "--maximum-hours", "12",
        "--stop-before-minutes", "30",
    ])
data_result = subprocess.run(data_command, cwd=SOURCE_DIR, check=False)
if data_result.returncode not in (0, 75):
    raise subprocess.CalledProcessError(data_result.returncode, data_command)
PREPROCESSING_PAUSED = data_result.returncode == 75
if PREPROCESSING_PAUSED:
    print("Preprocessing paused after a durable source-file checkpoint; training is deferred to the next session.")


In [ ]:
if PREPROCESSING_PAUSED:
    print("Skipping training in this session because preprocessing will resume first.")
else:
    train_command = [
    sys.executable, str(SOURCE_DIR / "train.py"),
    "--config", str(SOURCE_DIR / "config/train.json"),
    "--prepared-data-dir", str(PREPARED_DIR),
    "--output-dir", str(RUNS_DIR),
    "--upload-checkpoints-to-s3",
    ]

    if os.environ.get("RUN_ID"):
        train_command.extend(["--run-id", os.environ["RUN_ID"]])
    result = subprocess.run(train_command, cwd=SOURCE_DIR, check=False)
    if result.returncode not in (0, 75):
        raise subprocess.CalledProcessError(result.returncode, train_command)
    if result.returncode == 75:
        print("Session paused only after a verified checkpoint; the watchdog may launch the next session.")
    else:
        print("Training reached iteration 100 and final reporting completed or remains durably retryable.")


In [ ]:
active_path = RUNS_DIR / "active_run.json"
if active_path.exists():
    active = json.loads(active_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "run_id": active.get("run_id"),
        "status": active.get("status"),
        "current_iteration": active.get("current_iteration"),
    }, indent=2))
else:
    print("No active run pointer was created.")
